In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.colors import to_rgba
import seaborn as sns
import os
import yaml
from sklearn.metrics import (r2_score, confusion_matrix, precision_score, recall_score, f1_score,
                             accuracy_score)
import datashader as ds
from datashader.mpl_ext import dsshow
from core.helpers import (get_config_file, get_data_out, get_data_computed_in, get_value_of_lost_load,
                         get_process_names, get_experiment_id, set_rwth_color, get_input_file)
from core.plotting_helpers import (get_latex_columnwidth, get_latex_textwidth, plot_custom_violin,
                                   get_grouped_positions, get_center_positions,
                                   binarize_lost_load, binarize_overcapacity)
from core.ts_scaling import get_ts_scaling_names, get_ts_scaling_key, get_scale_mode_ts, get_ts_scaling_dict
from sm.surrogate_model import SurrogateModelNN
from sm.surrogates_feasible_space import Feasibility_LP
from sm.sado import (CostPredictorNN, FeasibilityPredictorNN, FeasibilityPredictorNoFP ,SADOMachineLearning, SADOConexHull)
from sm.routines import combine_operational_and_max_overcapacity_results, add_regret_to_sado_results

PROJECT_FILE = 'project_ID14_5.yaml'
CASE = 'JP1'
ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
DATA_IN = os.path.join(ROOT, 'data', CASE, 'Input_Data')
CONFIG = get_config_file(DATA_IN, PROJECT_FILE)

DATA_COMPUTED_IN = get_data_computed_in(DATA_IN, PROJECT_FILE)
DATA_OUT         = get_data_out(DATA_IN, PROJECT_FILE)

print(f"Experiment ID: '{get_experiment_id(DATA_IN, PROJECT_FILE)}'")

# **(I)** Train **OPEX** and **feasibility** predictors

## **Step I.1**: Generate Training Data Inputs

The process of generating data can be found in demo_Sampling.ipynb in Section **3. Demonstration of sampling different timeseries and design capacities**.

Gives: 
* ```operational_opt_var_ts_lhs_sample_100000.csv```
* ```operational_opt_var_ts_ts_ignore.yaml```

## **Step I.2:** Calculate Training Data Outputs
Is Done on the HPC with ```script_JP1_operation.sh```.

Gives:
* ```op_var_100000.csv``` (with results of opterational optimization)
* ```op_var_max_overcapacity_100000.csv``` (with results of max overcapacity optimization)


In [ ]:
# Or not on the HPC:
from sm.routines import operational_optimization_var_ts, generate_aggregated_input_data
from core.helpers import UniqueKeyLoader

TS_IGNORE_FILE = 'operational_opt_var_ts_ts_ignore.yaml'
CSV_INPUT_NAME = 'operational_opt_var_ts_lhs_sample_100000.csv'

# Generate aggregated input data:
generate_aggregated_input_data(DATA_IN, PROJECT_FILE)

# Send timeseries_ignore to all workers
with open(os.path.join(DATA_COMPUTED_IN, TS_IGNORE_FILE), 'rb') as f:
    timeseries_ignore = yaml.load(f, Loader=UniqueKeyLoader)

df_in = pd.read_csv(os.path.join(DATA_COMPUTED_IN, CSV_INPUT_NAME))

# Operational optimization
df_operation = operational_optimization_var_ts(
    df_in = df_in[5000:5005],
    data_in_path=DATA_IN,
    optimization_type='operation', #'max_overcapacity',
    project_file=PROJECT_FILE,
    timeseries_ignored= timeseries_ignore,
    debug=True # True: Print solver log, for each optimization: export output_dict and result_plot
)
# df_operation.to_csv(os.path.join(DATA_OUT, 'op_var_10000.csv'), index=False)

# Max Overcapacity optimization
df_max_overcapacity = operational_optimization_var_ts(
    df_in = df_in,
    data_in_path=DATA_IN,
    optimization_type= 'max_overcapacity',
    project_file=PROJECT_FILE,
    timeseries_ignored= timeseries_ignore,
    debug=False # True: Print solver log, for each optimization: export output_dict and result_plot
)
df_max_overcapacity.to_csv(os.path.join(DATA_OUT, 'op_var_max_overcapacity_10000.csv'), index=False)

## Revisions: Plots ENS and OC distribution

In [ ]:
n_days = [5, 10, 50, 100]

plt.close('all')

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
fig, axs = plt.subplots(
    len(n_days),
    2,
    figsize=(6.593453438, 6.5),
    constrained_layout=True
)

for i, n_day in enumerate(n_days):
    project_file = f"project_ID14_{n_day}.yaml"
    data_out = get_data_out(DATA_IN, project_file)
    # df_train = pd.read_csv(os.path.join(data_out, 'op_var_100000_train.csv'))
    df_test = pd.read_csv(os.path.join(data_out, 'op_var_100000_test.csv'))
    df_combined = df_test #pd.concat([df_train, df_test], ignore_index=True)

    # plot histogram of OC in left column without seaborn
    bins = np.linspace(-100, 400, 21)
    axs[i, 1].hist(df_combined['overcapacity']*100, bins=bins[4:], color=set_rwth_color('blue'), edgecolor='black', label='feasible', zorder=2)
    axs[i, 1].hist(df_combined['overcapacity']*100, bins=bins[:5], color=set_rwth_color('black', 50), edgecolor='black', label='infeasible', zorder=2)
    axs[i, 1].set_title(f'{n_day} days')
    if i == len(n_days)-1:
        axs[i, 1].set_xlabel(r'overcapacity $\varepsilon^\mathrm{OC}$ in \%')
    axs[i, 1].set_ylabel('count')
    axs[i, 1].grid(axis='y')
    axs[i, 1].set_yscale('log')
    axs[i, 1].set_ylim(1, 2e4)
    axs[i, 1].set_yticks([1, 10, 100, 1000, 10000])
    if i == 0:
        axs[i, 1].legend()

    # plot energy not served in right column without seaborn
    # one bin should only contain ens = 0
    # the others should be 20 bins between the >0 and maximum of ens, excluding 0
    max_ens =df_combined['lost_load'].max()/1000
    bins =  [0] + list(np.linspace(0.01, max_ens, 20))
    axs[i, 0].hist(df_combined['lost_load']/1000, bins=bins[:2] , color=set_rwth_color('orange'), edgecolor=set_rwth_color('orange'),
                   linewidth=2, label=r'feasible', zorder=2.1)
    axs[i, 0].hist(df_combined['lost_load']/1000, bins=bins[1:], color=set_rwth_color('black', 50), edgecolor='black', label='infeasible', zorder=2)
    axs[i, 0].set_title(f'{n_day} days')
    if i == len(n_days)-1:
        axs[i, 0].set_xlabel(r'total energy not supplied $W^\mathrm{ENS}$ in MWh')
    axs[i, 0].set_ylabel('count')
    axs[i, 0].set_yscale('log')
    axs[i, 0].set_ylim(0.8, 2e4)
    axs[i, 0].set_yticks([1, 10, 100, 1000, 10000])
    axs[i, 0].ticklabel_format(axis='x', style='plain')
    axs[i, 0].grid(axis='y', which='both')
    if i == 0:
        axs[i, 0].legend()
# Save plot
os.makedirs(os.path.join(DATA_OUT, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(DATA_OUT, "plots", "oc_histograms.pdf"),
    dpi=300,
    bbox_inches='tight'
)
print(DATA_OUT)

In [ ]:
n_days = [100]

plt.close('all')

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
fig, axs = plt.subplots(
    len(n_days),
    2,
    figsize=(6.593453438, 6.5/4),
    constrained_layout=True
)

project_file = f"project_ID14_{n_day}.yaml"
data_out = get_data_out(DATA_IN, project_file)
# df_train = pd.read_csv(os.path.join(data_out, 'op_var_100000_train.csv'))
df_test = pd.read_csv(os.path.join(data_out, 'op_var_100000_test.csv'))
df_combined = df_test #pd.concat([df_train, df_test], ignore_index=True)

# plot energy not served in right column without seaborn
# one bin should only contain ens = 0
# the others should be 20 bins between the >0 and maximum of ens, excluding 0
max_ens =df_combined['lost_load'].max()/1000
bins =  [0] + list(np.linspace(0.01, max_ens, 20))
axs[0].hist(df_combined['lost_load']/1000, bins=bins[:2] , color=set_rwth_color('orange'), edgecolor=set_rwth_color('orange'),
                linewidth=2, label=r'feasible', zorder=2.1)
axs[0].hist(df_combined['lost_load']/1000, bins=bins[1:], color=set_rwth_color('black', 50), edgecolor='black', label='infeasible', zorder=2)
# axs[1].set_title(f'{n_day} days')
axs[0].set_xlabel(r'total energy not supplied $W^\mathrm{ENS}$ in MWh')
axs[0].set_ylabel('count')
axs[0].set_yscale('log')
axs[0].set_ylim(0.8, 2e4)
axs[0].set_yticks([1, 10, 100, 1000, 10000])
axs[0].ticklabel_format(axis='x', style='plain')
axs[0].grid(axis='y', which='both')
axs[0].legend()

# plot histogram of OC in left column without seaborn
bins = np.linspace(-100, 400, 21)
axs[1].hist(df_combined['overcapacity']*100, bins=bins[4:], color=set_rwth_color('blue'), edgecolor='black', label='feasible', zorder=2)
axs[1].hist(df_combined['overcapacity']*100, bins=bins[:5], color=set_rwth_color('black', 50), edgecolor='black', label='infeasible', zorder=2)
# axs[0].set_title(f'{n_day} days')
axs[1].set_xlabel(r'overcapacity $\varepsilon^\mathrm{OC}$ in \%')
axs[1].set_ylabel('count')
axs[1].grid(axis='y')
axs[1].set_yscale('log')
axs[1].set_ylim(1, 2e4)
axs[1].set_yticks([1, 10, 100, 1000, 10000])
axs[1].legend()
# Save plot
os.makedirs(os.path.join(DATA_OUT, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(DATA_OUT, "plots", "oc_histograms_100_days.pdf"),
    dpi=300,
    bbox_inches='tight'
)
print(DATA_OUT)

## **Step I.3:** Train Surrogate Models

Is done on the HPC with  ```sscript_JP1_sm_training.sh```.

Or can be done here (shown below).
Gives:
* NN_OPEX
* NN_lostload
* NN_overcapacity

In [ ]:
# Load data
filename = 'op_var_1e5.csv'
df_train_raw = pd.read_csv(os.path.join(DATA_OUT, filename))

# Load overcapacity data
df_overcapacity = pd.read_csv(os.path.join(DATA_OUT, 'op_var_max_overcapacity_1e5.csv'))

# Overwrite the 'overcapacity' column in df_train_raw with the values from df_overcapacity
df_train_raw['overcapacity'] = df_overcapacity['overcapacity']

SM_CONFIG_NAME = 'ANN_4L'

In [ ]:
# value of loss load dict (with products as keys):
voLL_dict = get_value_of_lost_load(DATA_IN, PROJECT_FILE)

# Calculate OPEX minus loss load for each product
df_train_raw['OPEX_woLL'] = df_train_raw['OPEX'] - sum(
    voLL_dict[product] * df_train_raw[f'lost_load_{product}']
    for product in voLL_dict.keys()
)
print(df_train_raw.columns)

test_size = CONFIG['surrogate_models'][SM_CONFIG_NAME]['test_size']
# Sanity check on test size:
if not (0 <= test_size < 1):
    raise ValueError(f"Test size must be between 0 and 1, got {test_size}")

# Split data into training and testing sets
train_size = int((1 - test_size) * len(df_train_raw))

# Randomly shuffle the data
df_train_raw_shuffle = df_train_raw.sample(frac=1, random_state=CONFIG['seed']).reset_index(drop=True).copy()

# Split the data
df_train = df_train_raw_shuffle.iloc[:train_size].copy(deep=True)
df_test = df_train_raw_shuffle.iloc[train_size:].copy(deep=True)

# Save the training and testing data
df_train.to_csv(os.path.join(DATA_OUT, filename.replace('.csv', '_train.csv')), index=False)
df_test.to_csv(os.path.join(DATA_OUT, filename.replace('.csv', '_test.csv')), index=False)

# print lengths
print(f"Training set length: {len(df_train)}, Testing set length: {len(df_test)}")


### Train model to predict OPEX_woLL (OPEX minus lostload)

In [ ]:
processes = get_process_names(
    DATA_IN, PROJECT_FILE, only_feasibility_affecting=False, storage_or_production='both')
ts_scaling_names = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=False)

sm_opex = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names = processes + ts_scaling_names,
    output_name='OPEX_woLL',
    sm_name='ANN_4L')
sm_opex.generate_train_val_test_sets(df_train, split_test_data=False)
sm_opex.train_model(checkpoint_path=os.path.join(DATA_OUT, 'NN_OPEX'))
sm_opex.save_model(output_path=os.path.join(DATA_OUT, 'NN_OPEX'), filename='NN_OPEX')
_ = sm_opex.plot_training_loss()

In [ ]:
# Continue Training

sm_opex.train_model()
sm_opex.save_model(filename="NN_2e5")
_ = sm_opex.plot_training_loss()

In [ ]:
processes = get_process_names(
    DATA_IN, PROJECT_FILE, only_feasibility_affecting=False, storage_or_production='both')
ts_scaling_names = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=False)

# Load Surrogate
sm_opex = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names =  processes + ts_scaling_names,
    output_name='OPEX_woLL',
    sm_name='ANN_4L')

sm_opex.setup_sm()
sm_opex.load_model(os.path.join(DATA_OUT, 'NN_opex'), 'NN_opex')
# Load specific parameter set:
# sm_opex.sm.load_params(f_params=os.path.join(DATA_OUT, 'NN_opex','params_397.pkl'))

In [ ]:
# Determine the R2 score on the test set
df_test['opex_wo_LL_predict'] = sm_opex.predict(df_test)

r2 = r2_score(df_test['OPEX_woLL'], df_test['opex_wo_LL_predict'])
print(r2)

### Train Surrogate to predict lost load:

In [ ]:
processes_af = get_process_names(
    DATA_IN, PROJECT_FILE, only_feasibility_affecting=True, storage_or_production='both')
ts_scaling_names_af = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)

sm_ll = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names = processes_af + ts_scaling_names_af,
    output_name='lost_load',
    sm_name='ANN_4L')

sm_ll.generate_train_val_test_sets(df_train, split_test_data=False)
sm_ll.train_model(checkpoint_path=os.path.join(DATA_OUT, 'NN_lostload'))
sm_ll.save_model(output_path=os.path.join(DATA_OUT, 'NN_lostload'), filename='NN_lostload')
_ = sm_ll.plot_training_loss()

In [ ]:
# Determine the R2 score on the test set
df_test['lost_load_predict'] = sm_ll.predict(df_test)

r2 = r2_score(df_test['lost_load'], df_test['lost_load_predict'])
print(r2)

### Train Surrogate to predict overcapacity:

In [ ]:
processes_af = get_process_names(
    DATA_IN, PROJECT_FILE, only_feasibility_affecting=True, storage_or_production='both')
ts_scaling_names_af = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)

sm_oc = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names = processes_af + ts_scaling_names_af,
    output_name='overcapacity',
    sm_name='ANN_4L')
sm_oc.generate_train_val_test_sets(df_train, split_test_data=False)
sm_oc.train_model(checkpoint_path=os.path.join(DATA_OUT, 'NN_overcapacity'))
sm_oc.save_model(output_path=os.path.join(DATA_OUT, 'NN_overcapacity'), filename='NN_overcapacity')
_ = sm_oc.plot_training_loss()

In [ ]:
# Determine the R2 score on the test set
df_test['overcapacity_predict'] = sm_oc.predict(df_test)

r2 = r2_score(df_test['overcapacity'], df_test['overcapacity_predict'])
print(r2)

In [ ]:
# Export df_test
df_test.to_csv(os.path.join(DATA_OUT, filename.replace('.csv', '_test_with_predictions.csv')), index=False)

### Train Surrogate to predict binary feasibility:

In [ ]:
from sm.surrogate_model import SurrogateModelNN_Binary

processes_af = get_process_names(
    DATA_IN, PROJECT_FILE, only_feasibility_affecting=True, storage_or_production='both')
ts_scaling_names_af = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)

df_train = pd.read_csv(os.path.join(DATA_OUT, 'op_var_100000_train.csv'))

sm_feasibility = SurrogateModelNN_Binary(
    DATA_IN,
    PROJECT_FILE,
    input_names = processes_af + ts_scaling_names_af,
    output_name='overcapacity',
    sm_name='ANN_feasibility')

sm_feasibility.generate_train_val_test_sets(df_train, split_test_data=False)
sm_feasibility.train_model()
sm_feasibility.save_model(output_path=os.path.join(DATA_OUT, 'NN_binary_feasibility'), filename='NN_binary_feasibility')
_ = sm_feasibility.plot_training_loss()

## **Step I.4** Test Surrogates:


### Classification

In [ ]:
# Define helper functions:

def compute_confusion(true_labels, pred_labels):

    # Compute confusion matrix
    cm = confusion_matrix(true_labels, pred_labels, labels=[1, 0])

    metrics = {
        'accuracy': accuracy_score(true_labels, pred_labels),
        'precision': precision_score(true_labels, pred_labels, zero_division=0),
        'recall': recall_score(true_labels, pred_labels, zero_division=0),
        'f1_score': f1_score(true_labels, pred_labels, zero_division=0)
    }
    # calc postive and negative predictive value
        # PPV = TP / (TP + FP)
        # NPV = TN / (TN + FN)
    metrics['PPV'] = cm[0, 0] / (cm[0, 0] + cm[1, 0]) if (cm[0, 0] + cm[1, 0]) > 0 else 0
    metrics['NPV'] = cm[1, 1] / (cm[1, 1] + cm[0, 1]) if (cm[1, 1] + cm[0, 1]) > 0 else 0

    return cm, metrics

In [ ]:
# load all three feasibility predictors:
ts_scaling_names_af = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)
processes_af = get_process_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True, storage_or_production='both')

sm_ll = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE, 
    input_names =  ts_scaling_names_af + processes_af,
    output_name='lost_load',
    sm_name='ANN_4L')

sm_ll.setup_sm()
sm_ll.load_model(os.path.join(DATA_OUT, 'NN_lostload'), 'NN_lostload')

sm_oc = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names = ts_scaling_names_af + processes_af,
    output_name='overcapacity',
    sm_name='ANN_4L')

sm_oc.setup_sm()
sm_oc.load_model(os.path.join(DATA_OUT, 'NN_overcapacity'), 'NN_overcapacity')

sm_feasibility = SurrogateModelNN_Binary(
    DATA_IN,
    PROJECT_FILE,
    input_names = ts_scaling_names_af + processes_af,
    output_name='overcapacity',
    sm_name='ANN_feasibility')

sm_feasibility.setup_sm()
sm_feasibility.load_model(os.path.join(DATA_OUT, 'NN_binary_feasibility'), 'NN_binary_feasibility')

vertex_df = pd.read_csv(os.path.join(DATA_OUT, "convex_hull_vertex_results_5000.csv"))
feasibility_lp = Feasibility_LP(DATA_IN, PROJECT_FILE, vertex_df)

In [ ]:
df_test = pd.read_csv(os.path.join(DATA_OUT, 'op_var_100000_test.csv'))

print(f"Test size: {len(df_test)}")
# Predict lost load and overcapacity
df_test['overcapacity_predict'] = sm_oc.predict(df_test)
df_test['lost_load_predict'] = sm_ll.predict(df_test)
df_test['feasibility_binary'] = sm_feasibility.predict(df_test)

df_test['feasibility_convexhull'], _ = feasibility_lp.predict_feasibility(df_test)

df_test.to_csv(os.path.join(DATA_OUT, 'op_var_100000_test_with_feasibility_predictions.csv' ), index=False)

In [ ]:
plt.close('all')

def plot_cm(cm, ax_labels, ax):
    total = cm.sum()
    percentages = cm / total * 100

    labels = np.array([
        [f'{cm[0, 0]}\n({percentages[0, 0]:.1f}\\%)', f'{cm[0, 1]}\n({percentages[0, 1]:.1f}\\%)'],
        [f'{cm[1, 0]}\n({percentages[1, 0]:.1f}\\%)', f'{cm[1, 1]}\n({percentages[1, 1]:.1f}\\%)']
    ])

    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues',
                xticklabels=ax_labels, yticklabels=ax_labels, ax=ax, cbar=False)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1,4, figsize=(10,3), constrained_layout=True)

# LOSTLOAD
cm_lostload, metrics_lostload = compute_confusion(
    true_labels = binarize_lost_load(df_test['lost_load']),
    pred_labels = binarize_lost_load(df_test['lost_load_predict'])
    )

plot_cm(
    cm = cm_lostload,
    # ax_labels= ['feasible\nENS $\\leq 0$', 'not feasible\nENS $> 0$'],
    ax_labels= ['feasible', 'infeasible'],
    ax=ax1)

ax1.set_xlabel('predicted')
ax1.set_ylabel('actual')
ax1.set_title('ENS predictor')

# OVERCAPACITY
cm_overcapacity, metrics_overcapacity = compute_confusion(
    true_labels = binarize_overcapacity(df_test['overcapacity']),
    pred_labels = binarize_overcapacity(df_test['overcapacity_predict'])
    )

plot_cm(
    cm = cm_overcapacity,
    # ax_labels= ['feasible\nOC $\\geq 0$', 'not feasible\nOC $< 0$'],
    ax_labels= ['feasible', 'infeasible'],
    ax=ax2)

ax2.set_xlabel('predicted')
ax2.set_ylabel('actual')
ax2.set_title('OC predictor')

# binary feasibility predictor
cm_binary, metrics_binary = compute_confusion(
    true_labels = binarize_overcapacity(df_test['overcapacity']),
    pred_labels = df_test['feasibility_binary'].values.astype(int)
    )
plot_cm(
    cm = cm_binary,
    # ax_labels= ['feasible', 'not feasible'],
    ax_labels= ['feasible', 'infeasible'],
    ax=ax3)
ax3.set_xlabel('predicted')
ax3.set_ylabel('actual')
ax3.set_title('binary predictor')



# CONVEXHULL
cm_convexhull, metrics_convexhull = compute_confusion(
    true_labels = binarize_lost_load(df_test['lost_load']),
    pred_labels = df_test['feasibility_convexhull'].values.astype(int)
    )

plot_cm(
    cm = cm_convexhull,
    # ax_labels= ['feasible', 'not feasible'],
    ax_labels= ['feasible', 'infeasible'],
    ax=ax4)

ax4.set_xlabel('predicted')
ax4.set_ylabel('actual')
ax4.set_title('CH predictor')

plt.savefig(
    os.path.join(DATA_OUT, "plots", "feasibility_predictor_confusion_matrices.pdf"),
    dpi=300,
    bbox_inches='tight'
)

# Print metrics:
print('EnergyNotServed:')
for key, val in metrics_lostload.items():
    print(f"\t{key}: {val:.4f}")

r2_ll = r2_score(df_test['lost_load'], df_test['lost_load_predict'])
print(f"\tR2: {r2_ll:.4f}")

print('\nOvercapacity:')
for key, val in metrics_overcapacity.items():
    print(f"\t{key}: {val:.4f}")

r2_oc = r2_score(df_test['overcapacity'], df_test['overcapacity_predict'])
print(f"\tR2: {r2_oc:.4f}")

print('\nBinary Feasibility Predictor:')
for key, val in metrics_binary.items():
    print(f"\t{key}: {val:.4f}")

r2_binary = r2_score(binarize_overcapacity(df_test['overcapacity']), df_test['feasibility_binary'].values.astype(int))
print(f"\tR2: {r2_binary:.4f}")

print('\nConvex Hull Feasibility Predictor:')
for key, val in metrics_convexhull.items():
    print(f"\t{key}: {val:.4f}")

r2_convexhull = r2_score(binarize_lost_load(df_test['lost_load']), df_test['feasibility_convexhull'].values.astype(int))
print(f"\tR2: {r2_convexhull:.4f}")

# save metrics as a table in a text file

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'PPV', 'NPV', 'R2'],
    'ENS Predictor': [metrics_lostload['accuracy'], metrics_lostload['precision'], metrics_lostload['recall'], metrics_lostload['f1_score'], metrics_lostload['PPV'], metrics_lostload['NPV'], r2_ll],
    'OC Predictor': [metrics_overcapacity['accuracy'], metrics_overcapacity['precision'], metrics_overcapacity['recall'], metrics_overcapacity['f1_score'], metrics_overcapacity['PPV'], metrics_overcapacity['NPV'], r2_oc],
    'Binary Predictor': [metrics_binary['accuracy'], metrics_binary['precision'], metrics_binary['recall'], metrics_binary['f1_score'], metrics_binary['PPV'], metrics_binary['NPV'], r2_binary],
    'CH Predictor': [metrics_convexhull['accuracy'], metrics_convexhull['precision'], metrics_convexhull['recall'], metrics_convexhull['f1_score'], metrics_convexhull['PPV'], metrics_convexhull['NPV'], r2_convexhull]
})

In [ ]:
# plot influence of oc_min on false positive (x axis) and false negative (yaxis)
# add colorscale for eps
# vary oc_min between 0 and 0.12 with 100 steps (including 0)
# PPV AND NPV
n_days = [5, 10, 50, 100]
OC_MAX = 5
OC_MIN = -2

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

# Plotting the results
plt.close('all')
fig, axs = plt.subplots(2,2, figsize=(3.296726719, 3.7), sharex=True, sharey=True, constrained_layout=True)

for i, n_day in enumerate(n_days):

    project_file = f"project_ID14_{n_day}.yaml"
    data_out = get_data_out(DATA_IN, project_file)
    ts_scaling_names_af = get_ts_scaling_names(DATA_IN, project_file, only_feasibility_affecting=True)
    processes_af = get_process_names(DATA_IN, project_file, only_feasibility_affecting=True, storage_or_production='both')

    # Load OC Surrogate
    sm_oc = SurrogateModelNN(
        DATA_IN,
        project_file,
        input_names = ts_scaling_names_af + processes_af,
        output_name='overcapacity',
        sm_name='ANN_4L')

    sm_oc.setup_sm()
    sm_oc.load_model(os.path.join(data_out, 'NN_overcapacity'), 'NN_overcapacity')

    # Load Test set:
    df_test = pd.read_csv(os.path.join(data_out, 'op_var_100000_test.csv'))

    df_test['overcapacity_predict'] = sm_oc.predict(df_test)

    oc_min_values = np.linspace(OC_MIN, OC_MAX, 99)# 103)
    TF = np.array([])  # Predicted feasible and actually feasible
    TI = np.array([])  # Predicted infeasible and actually infeasible
    FF = np.array([])  # Predicted feasible but actually not feasible 
    FI = np.array([]) # Predicted not feasible but actually feasible

    FIR = np.array([]) 
    FFR = np.array([])
    FDR = np.array([]) 
    FOR = np.array([])

    for oc_min in oc_min_values:
        # Binarize predictions with current oc_min
        pred_labels = binarize_overcapacity(df_test['overcapacity_predict']*100, oc_min=oc_min)
        
        # Compute confusion matrix
        cm = confusion_matrix(binarize_overcapacity(df_test['overcapacity']*100), pred_labels, labels=[1, 0])
        
        # Append false positives and negatives
        TF = np.append(TF, cm[0, 0])
        TI = np.append(TI, cm[1, 1])
        FF = np.append(FF, cm[1, 0])
        FI = np.append(FI, cm[0, 1])

    FIR = FI / (TF + FI) * 100 # False infeasible rate
    FFR = FF / (TI + FF) * 100 # False feasible rate
    FDR = FF / (TF + FF) * 100 # False Discovery rate. 
    FOR = FI / (TI + FI) * 100 # False Omission rate
    PPV = 100 - FDR # Positive predictions rate
    NPV = 100 - FOR # Negative predictions rate
    # scatter OC
    ax = axs.flatten()[i]
    sc = ax.scatter(NPV, PPV, c=oc_min_values,
                cmap='plasma', s=7, zorder=99)

    if i in [2,3]:
        ax.set_xlabel(r'NPV in \%')
    if i % 2 == 0:
        ax.set_ylabel(r'PPV in \%')

    for oc_min_value in range(OC_MIN, OC_MAX + 1):
        index = np.where(oc_min_values == oc_min_value)[0][0]

        ax.text(
            NPV[index] +0.3, PPV[index]+ 0.08,
            rf"{oc_min_value}", zorder=101#, fontsize=9 
            )
        ax.scatter(NPV[index], PPV[index], color='white', marker='o', edgecolor='k', s=10, zorder=100)

    ax.set_title(f"{n_day} days")

cbar = fig.colorbar(sc, ax=axs, orientation='horizontal', location='bottom', label=r'$\varepsilon^\mathrm{OC}_\mathrm{min}$ in \%')

for ax in axs.flatten():
    ax.set_ylim(ax.get_ylim()[0], 100.4)
    ax.set_xlim(90, 102)

    ax.xaxis.set_minor_locator(MultipleLocator(1))
    ax.xaxis.grid(True, which='major')
    ax.xaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
    ax.yaxis.set_minor_locator(MultipleLocator(0.2))
    ax.yaxis.set_major_locator(MultipleLocator(1.0))
    ax.yaxis.grid(True, which='major')
    ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
# plt.tight_layout()

# Save plot:
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_oc_classification.pdf"), dpi=300, bbox_inches='tight')

In [ ]:
# plot influence of oc_min on false positive (x axis) and false negative (yaxis)

# vary oc_min between 0 and 0.12 with 100 steps (including 0)
n_days = [5, 10, 50, 100]



plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

# Plotting the results
plt.close('all')
fig, ax = plt.subplots(1, 1, figsize=(3.296726719, 2.037), constrained_layout=True)

positions = {
    5:   {'x': 5, 'y': 1},
    10:  {'x': 65,  'y': 1},
    50:  {'x': 10,  'y': 3},
    100: {'x': 20, 'y': 2},
}
for i, n_day in enumerate(n_days):
    project_file = f"project_ID14_{n_day}.yaml"
    data_out = get_data_out(DATA_IN, project_file)
    ts_scaling_names_af = get_ts_scaling_names(DATA_IN, project_file, only_feasibility_affecting=True)
    processes_af = get_process_names(DATA_IN, project_file, only_feasibility_affecting=True, storage_or_production='both')

    # Load LL Surrogate
    sm_ll = SurrogateModelNN(
        DATA_IN,
        project_file, 
        input_names =  ts_scaling_names_af + processes_af,
        output_name='lost_load',
        sm_name='ANN_4L')

    sm_ll.setup_sm()
    sm_ll.load_model(os.path.join(data_out, 'NN_lostload'), 'NN_lostload')

    # Load Test set:
    df_test = pd.read_csv(os.path.join(data_out, 'op_var_100000_test.csv'))


    # Predict LL:
    df_test['lost_load_predict'] = sm_ll.predict(df_test)
    pred_labels = binarize_lost_load(df_test['lost_load_predict'])
    
    # Compute confusion matrix
    cm = confusion_matrix(binarize_lost_load(df_test['lost_load']), pred_labels, labels=[1, 0])

    # Append false positives and negatives
    TF = cm[0, 0] / len(df_test) *100  # Predicted feasible and actually feasible
    TI = cm[1, 1]/ len(df_test) *100  # Predicted infeasible and actually infeasible
    FF = cm[1, 0] / len(df_test) *100  # Predicted feasible but actually not feasible 
    FI = cm[0, 1] / len(df_test) * 100 # Predicted not feasible but actually feasible

    FIR = FI / (TF + FI) * 100# False infeasible rate
    FFR = FF / (TI + FF) * 100# False feasible rate
    FOR = FI / (TI + FI) * 100 # False Omission rate
    NPV = 100 - FOR

    if (FF == 0) and (TF == 0):
        
        FDR = 100
        PPV = 96
        marker = '^'
        color = 'grey'
        ax.vlines(NPV, 96, 100, colors='k', linestyles='--')
        ax.text(40, 98, f"{n_day} days", ha='left', va='center')
    else:
        FDR = FF / (TF + FF) * 100 # False Discovery rate.
        marker = 'o'
        color = 'k'
        PPV = 100 - FDR
        ax.scatter(NPV , PPV, c=color, s=25, zorder=1000, marker=marker, clip_on=False)
        ax.text(90, PPV, f"{n_day} days", ha='right', va='center')
    

ax.set_xlabel(r'NPV in \%')

ax.xaxis.grid(True, which='major')
ax.xaxis.set_minor_locator(MultipleLocator(5))
ax.xaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)


ax.yaxis.grid(True, which='major')
ax.yaxis.set_major_locator(MultipleLocator(1))
ax.yaxis.set_minor_locator(MultipleLocator(0.25))
ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
ax.set_ylabel(r'PPV in \%')
#ax.set_title(f"ENS Predictor")

ax.set_xlim(30, 100)
ax.set_ylim(96, 100)
# plt.tight_layout()

# Save plot:
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_ens_classification.pdf"), dpi=300, bbox_inches='tight')

### Classification Revisions

In [ ]:


# Plot cdf-style plot of absolute true overcapacity. Not normed to 1, but with the absolute number of samples on the y-axis.
plt.close('all')
fig, ax = plt.subplots(1, 1, figsize=(6, 4), constrained_layout=True)
# Sort the data
df_sorted = df_test.sort_values(by='overcapacity_abs')

# Compute the cumulative sum of samples
df_sorted['cumulative_samples'] = np.arange(1, len(df_sorted) + 1)
ax.plot(df_sorted['overcapacity_abs'], df_sorted['cumulative_samples'], drawstyle='steps-post')

ax.set_xlabel(r'Absolute true overcapacity in \%')
ax.set_ylabel('Number of samples')
ax.set_title('Cumulative distr. of abs. true overcapacity')
ax.xaxis.grid(True, which='major')

# log scale for x-axis
ax.set_xscale('log')

ax.set_xlim(0.01, 400)

# log scale for y-axis
ax.set_yscale('log')
ax.set_ylim(1, len(df_test)*2)

ax.yaxis.grid(True, which='major')


In [ ]:
from core.plotting_helpers import create_piecewise_linear_scale

# Plot conditional, binned NPV and PPV of predictors over true absolute overcapacity.
# Bins are logarithmically distributed, like a histogram with log-spaced bins.

df_test = pd.read_csv(
    os.path.join(DATA_OUT, 'op_var_100000_test_with_feasibility_predictions.csv')
)

# Convert id to int and sort
df_test['id'] = df_test['id'].astype(int)
df_test = df_test.sort_values(by='id')

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

# Absolute true overcapacity in percentage points
df_test['overcapacity_abs'] = df_test['overcapacity'].abs() * 100

plt.close('all')

colors = {
    'LL': 'orange',
    'OC': 'blue',
    'OC_0.05': 'green',
    'OC_0.1': 'petrol',
    'CH': 'magenta',
    'BM': 'black',
    'noFP': 'red',
    'binary': 'red',
}

labels = {
    'LL': r'ENS predictor',
    'OC': r'OC predictor',
    'OC_0.05': r'OC, 5\%',
    'OC_0.1': r'OC, 10\%',
    'CH': 'CH predictor',
    'BM': 'BM',
    'noFP': 'none',
    'binary': 'binary predictor',
}

zorder = {'LL': 5,'OC': 4,'OC_0.05': 4.1,'OC_0.1': 4.2,'CH': 3, 'binary': 2}
# increase markersize from last to first predictor, to make sure they are visible:
markersize = {'LL':4,'OC':4,'OC_0.05': 3.5,'OC_0.1': 5.3,'CH':4, 'binary': 4}

# -------------------------------------------------------------------------
# Logarithmic bins
# -------------------------------------------------------------------------
# Log-spaced bin edges from 10^-1 to 10^2.5
log_edges = np.logspace(-1, 2.5, num=4 * 4 + 1 - 2)

# Add one underflow bin: [0, first_log_edge)
# This keeps samples with overcapacity_abs == 0.
bin_lows = np.r_[0.0, log_edges[:-1]]
bin_highs = log_edges

# Positive plotting positions for log-x axis
ratio = log_edges[1] / log_edges[0]

bin_centers = np.r_[
    log_edges[0] / np.sqrt(ratio),
    np.sqrt(log_edges[:-1] * log_edges[1:])
]

# Histogram-style bar positions
plot_left_edges = np.r_[
    log_edges[0] / ratio,
    log_edges[:-1]
]

plot_widths = bin_highs - plot_left_edges


def safe_percent(num, den):
    out = np.full_like(num, np.nan, dtype=float)
    np.divide(num, den, out=out, where=den != 0)
    return 100 * out


def get_y_pred(df_subset: pd.DataFrame, feas_pred: str):
    if feas_pred == 'LL':
        return binarize_lost_load(df_subset['lost_load_predict'])

    elif feas_pred == 'OC':
        return binarize_overcapacity(
            df_subset['overcapacity_predict'],
            oc_min=0.0
        )

    elif feas_pred == 'OC_0.05':
        return binarize_overcapacity(
            df_subset['overcapacity_predict'],
            oc_min=0.05
        )

    elif feas_pred == 'OC_0.1':
        return binarize_overcapacity(
            df_subset['overcapacity_predict'],
            oc_min=0.1
        )

    elif feas_pred == 'CH':
        return df_subset['feasibility_convexhull']

    elif feas_pred == 'binary':
        return df_subset['feasibility_binary']

    else:
        raise ValueError(f"Unknown feas_pred: '{feas_pred}'")


def get_binned_PPV_NPV(
    bin_lows: np.ndarray,
    bin_highs: np.ndarray,
    df_test: pd.DataFrame,
    feas_pred: str
):
    TP = []
    FP = []
    FN = []
    TN = []
    samples = []

    for i, (bin_low, bin_high) in enumerate(zip(bin_lows, bin_highs)):

        # Last bin includes the upper edge
        if i == len(bin_lows) - 1:
            mask = (
                (df_test['overcapacity_abs'] >= bin_low) &
                (df_test['overcapacity_abs'] <= bin_high)
            )
        else:
            mask = (
                (df_test['overcapacity_abs'] >= bin_low) &
                (df_test['overcapacity_abs'] < bin_high)
            )

        df_subset = df_test.loc[mask]
        samples.append(len(df_subset))

        if len(df_subset) == 0:
            TP.append(0)
            FP.append(0)
            FN.append(0)
            TN.append(0)
            continue

        y_true = binarize_overcapacity(df_subset['overcapacity'])
        y_pred = get_y_pred(df_subset, feas_pred)

        cm = confusion_matrix(y_true, y_pred, labels=[1, 0])

        TP.append(cm[0, 0])
        FP.append(cm[1, 0])
        FN.append(cm[0, 1])
        TN.append(cm[1, 1])

    TP = np.array(TP)
    FP = np.array(FP)
    FN = np.array(FN)
    TN = np.array(TN)
    samples = np.array(samples)

    return {
        'TP': TP,
        'FP': FP,
        'FN': FN,
        'TN': TN,
        'samples': samples,
        'PPV': safe_percent(TP, TP + FP),
        'NPV': safe_percent(TN, TN + FN)
    }


fig, axs = plt.subplots(
    2,
    1,
    figsize=(6.593453438, 4),
    constrained_layout=True,
    height_ratios=[1, 2]
)

feas_preds = ['LL', 'OC', 'OC_0.05', 'OC_0.1', 'CH', 'binary']

feas_dicts = {
    feas_pred: get_binned_PPV_NPV(
        bin_lows,
        bin_highs,
        df_test,
        feas_pred
    )
    for feas_pred in feas_preds
}

# -------------------------------------------------------------------------
# Top plot: histogram-style number of samples per log bin
# -------------------------------------------------------------------------
sample_counts = feas_dicts[feas_preds[0]]['samples']

axs[0].bar(
    plot_left_edges,
    sample_counts,
    width=plot_widths,
    align='edge',
    color=set_rwth_color('black', 25),
    edgecolor='black',
    linewidth=0.6,
    # zorder=5
)

axs[0].set_yscale('log')
axs[0].set_ylim(1, max(10, sample_counts.max() * 2))
axs[0].set_ylabel('number of samples')

axs[0].set_xscale('log')
axs[0].yaxis.grid(True, which='major')
axs[0].xaxis.grid(True, which='major')
axs[0].set_yticks([1, 10, 100, 1000, 10000])

# -------------------------------------------------------------------------
# Bottom plot: PPV and NPV per bin, not cumulative
# -------------------------------------------------------------------------
for feas_pred, feas_dict in feas_dicts.items():

    axs[1].plot(
        bin_centers,
        feas_dict['PPV'],
        marker='o',
        markersize=4,
        linestyle='-',
        color=set_rwth_color(colors[feas_pred]),
        zorder=zorder[feas_pred]
    )

    axs[1].plot(
        bin_centers,
        feas_dict['NPV'],
        marker='x',
        # markersize=markersize[feas_pred],
        linestyle=(0, (1, 1)), # densely dotted
        color=set_rwth_color(colors[feas_pred]),
        zorder=zorder[feas_pred]
    )

axs[1].yaxis.grid(True, which='major')
axs[1].xaxis.grid(True, which='major')

axs[1].yaxis.set_major_locator(MultipleLocator(2))

axs[1].set_ylabel('PPV and NPV in \\%', color='black')
axs[1].set_xlabel(
    r'distance to feasibility boundary $|\varepsilon^\mathrm{OC}|$ in \%',
    fontsize=10
)

for ax in axs:
    ax.set_xscale('log')
    ax.set_xlim(plot_left_edges[0]*0.75, 5e2)

VSPLIT = 90

scale_name: str = create_piecewise_linear_scale(
    vmin=-4.5,
    vsplit=VSPLIT,
    vmax=101,
    split_pos=0.5
)

axs[1].set_yscale(scale_name)

axs[1].set_yticks(np.concatenate([
    np.arange(0, 81, 20),
    np.arange(90, 100.1, 2)
]))

axs[1].set_ylim(-4.5, 100.5)

# Legend
for feas_pred in feas_preds:
    axs[1].plot(
        [],
        [],
        color=set_rwth_color(colors[feas_pred]),
        linestyle='-',
        label=f'{labels[feas_pred]}',
        markersize=4
    )

axs[1].plot(
    [],
    [],
    color='black',
    linestyle='-',
    label='PPV',
    marker='o',
    markersize=4
)

axs[1].plot(
    [],
    [],
    color='black',
    linestyle=(0, (1, 1)), # densely dotted
    label='NPV',
    marker='x'
)

axs[1].hlines(
    VSPLIT,
    plot_left_edges[0]*0.1,
    1e3,
    colors=set_rwth_color('black', 75),
    linestyles='--',
    label='scale split'
)

# ax[1] x ticks should be formatted as 0.01, 0.1, 1, 10, 100, not in scientific notation
# avoid rounding 0.1 to 0 in the x-axis ticks:
from matplotlib.ticker import FuncFormatter
def log_tick_formatter(x, pos):
    if x >= 1:
        return f'{int(x)}'
    else:
        return f'{x:.1f}'
axs[1].xaxis.set_major_formatter(FuncFormatter(log_tick_formatter))

axs[1].legend(loc='lower right')

axs[0].set_xticklabels([])

# Save plot
os.makedirs(os.path.join(DATA_OUT, "plots"), exist_ok=True)

n_days = get_config_file(DATA_IN, PROJECT_FILE)['time_series_aggregation']['noTypicalPeriods']
plt.savefig(
    os.path.join(DATA_OUT, "plots", f"fig11b_{n_days}days_binary.pdf"),
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
n_days = [5, 10, 50, 100]

# make same plot as above (without the histogram).
# Each numer of days in a different subplot, with the same x and y limits.
# 2 x 2 subplots, with shared x and y axes.
# split y axis at 90%, with a break mark.


labels = {
    'LL': r'ENS predictor',
    'OC': r'OC predictor',
    'OC_0.05': r'OC, 5\%',
    'OC_0.1': r'OC, 10\%',
    'CH': 'CH predictor',
    'BM': 'BM',
    'noFP': 'none'
}

plt.close('all')

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
fig, axs = plt.subplots(
    2,
    2,
    figsize=(6.593453438, 5),
    constrained_layout=True
)
feas_preds = [ 'LL', 'OC', 'CH'] 

for i, n_day in enumerate(n_days):
    project_file = f"project_ID14_{n_day}.yaml"
    data_out = get_data_out(DATA_IN, project_file)
    ts_scaling_names_af = get_ts_scaling_names(DATA_IN, project_file, only_feasibility_affecting=True)
    processes_af = get_process_names(DATA_IN, project_file, only_feasibility_affecting=True, storage_or_production='both')

    df_test = pd.read_csv(os.path.join(data_out, 'op_var_100000_test_with_feasibility_predictions.csv'))

    # Convert id to int and sort
    df_test['id'] = df_test['id'].astype(int)
    df_test = df_test.sort_values(by='id')

    # Absolute true overcapacity in percentage points
    df_test['overcapacity_abs'] = df_test['overcapacity'].abs() * 100


    for feas_pred in feas_preds:
        feas_dict = get_binned_PPV_NPV(
            bin_lows,
            bin_highs,
            df_test,
            feas_pred
        )

        ax = axs.flatten()[i]
        ax.plot(
            bin_centers,
            feas_dict['PPV'],
            marker='o',
            markersize=4,
            linestyle='-',
            color=set_rwth_color(colors[feas_pred]),
            zorder=zorder[feas_pred]
        )

        ax.plot(
            bin_centers,
            feas_dict['NPV'],
            marker='x',
            linestyle= (0, (1, 1)), # densely dotted
            color=set_rwth_color(colors[feas_pred]),
            zorder=zorder[feas_pred]
        )
        ax.set_title(f"{n_day} days")
        ax.yaxis.grid(True, which='major')
        ax.xaxis.grid(True, which='major')
        ax.yaxis.set_major_locator(MultipleLocator(2))
        ax.set_xscale('log')
        ax.set_xlim(plot_left_edges[0]*0.75, 5e2)
VSPLIT = 90
scale_name: str = create_piecewise_linear_scale(
    vmin=-4.5,
    vsplit=VSPLIT,
    vmax=101,
    split_pos=0.5
)
for ax in axs.flatten():
    ax.set_yscale(scale_name)
    ax.set_yticks(np.concatenate([
        np.arange(0, 81, 20),
        np.arange(90, 100.1, 2)
    ]))
    ax.set_ylim(-4.5, 100.5)
    ax.hlines(
        VSPLIT,
        plot_left_edges[0]*0.1,
        1e3,
        colors=set_rwth_color('black', 75),
        linestyles='--'
    )
# Legend
handles = []
for feas_pred in feas_preds:
    handles.append(
        plt.Line2D(
            [],
            [],
            color=set_rwth_color(colors[feas_pred]),
            linestyle='-',
            label=f'{labels[feas_pred]}',
            markersize=4
        )
    )
handles.append(
    plt.Line2D(
        [],
        [],
        color='black',
        linestyle='-',
        label='PPV',
        marker='o',
        markersize=4
    )
)
handles.append(
    plt.Line2D(
        [],
        [],
        color='black',
        linestyle=(0, (1, 1)), # densely dotted
        label='NPV',
        marker='x',
        markersize=4
    )
)
handles.append(
    plt.Line2D(
        [],
        [],
        color=set_rwth_color('black', 75),
        linestyle='--',
        label='scale split'
    )
)
axs[0,0].legend(
    handles=handles,
    loc='lower right',
    fontsize=8
)

for ax in [axs[1, 0], axs[1, 1]]:
    ax.set_xlabel(
        r'distance to feasibility boundary $|\varepsilon^\mathrm{OC}|$ in \%',
        fontsize=10
    )
for ax in [axs[0, 0], axs[1, 0]]:
    ax.set_ylabel('PPV and NPV in \\%', color='black', fontsize=10)

from matplotlib.ticker import FuncFormatter
def log_tick_formatter(x, pos):
    if x >= 1:
        return f'{int(x)}'
    else:
        return f'{x:.1f}'

for ax in axs.flatten():
    ax.xaxis.set_major_formatter(FuncFormatter(log_tick_formatter))

# Save plot
os.makedirs(os.path.join(DATA_OUT, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(DATA_OUT, "plots", f"abs_eps_SI.pdf"),
    dpi=300,
    bbox_inches='tight'
)
print(DATA_OUT)

In [ ]:
from core.plotting_helpers import create_piecewise_linear_scale

# Plot conditional and cumulative NPV and PPV of ENS predictor over true absolute overcapacity.
# For this, we use logarithmic linspace on abs true overcapacity
df_test = pd.read_csv(os.path.join(DATA_OUT, 'op_var_100000_test_with_feasibility_predictions.csv'))

# convert id to int
df_test['id'] = df_test['id'].astype(int)
# sort by id
df_test = df_test.sort_values(by='id')

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

# add absolute value of true overcapacity 
df_test['overcapacity_abs'] = df_test['overcapacity'].abs() * 100 # in percentage points
df_sorted['cumulative_samples'] = np.arange(1, len(df_sorted) + 1)

plt.close('all')

colors = {
    'LL': 'orange',
    'OC': 'blue',
    'OC_0.05': 'green',
    'OC_0.1': 'petrol',
    'CH': 'magenta',
    'BM': 'black',
    'noFP': 'red'
    }

labels = {
    'LL': r'ENS predictor',
    'OC': r'OC predictor',#, 0\%',
    'OC_0.05': r'OC, 5\%',
    'OC_0.1': r'OC, 10\%',
    'CH': 'CH predictor',
    'BM': 'BM',
    'noFP': 'none'
}
zorder = {'LL': 5,
          'OC': 4,
          'OC_0.05': 4,
          'OC_0.1': 4,
          'CH':3}
# use logarithmic linspace from 0.01 to 1000
oc_vals = np.logspace(-1,  2.5, num=4*4+1-2)

fig, axs = plt.subplots(2, 1, figsize=(6.593453438, 4), constrained_layout=True, height_ratios=[1, 2])

def get_cumulative_PPV_NPV(oc_vals:np.array, df_test:pd.DataFrame, feas_pred:str):

    TP_cumulative = []
    FP_cumulative = []
    FN_cumulative = []
    TN_cumulative = []

    for oc_val in oc_vals:
        df_subset = df_test[df_test['overcapacity_abs'] <= oc_val]

        if feas_pred == 'LL':
            y_pred = binarize_lost_load(df_subset['lost_load_predict'])

        elif feas_pred == 'OC':
            y_pred = binarize_overcapacity(df_subset['overcapacity_predict'], oc_min= 0.0)

        elif feas_pred == 'OC_0.05':
            y_pred = binarize_overcapacity(df_subset['overcapacity_predict'], oc_min= 0.05)

        elif feas_pred == 'OC_0.1':
            y_pred = binarize_overcapacity(df_subset['overcapacity_predict'], oc_min= 0.01)

        elif feas_pred == 'CH':
            y_pred = df_subset['feasibility_convexhull']

        else:
            raise ValueError(f"Unkown feas_pred: '{feas_pred}'")

        cm = confusion_matrix(binarize_overcapacity(df_subset['overcapacity']), y_pred, labels=[1, 0])

        TP_cumulative.append(cm[0, 0])
        FP_cumulative.append(cm[1, 0])
        FN_cumulative.append(cm[0, 1])
        TN_cumulative.append(cm[1, 1])

    # Convert to numpy arrays
    TP_cumulative = np.array(TP_cumulative)
    FP_cumulative = np.array(FP_cumulative)
    FN_cumulative = np.array(FN_cumulative)
    TN_cumulative = np.array(TN_cumulative)

    return {
        'TP_cumulative' : TP_cumulative,
        'FP_cumulative' : FP_cumulative,
        'FN_cumulative' : FN_cumulative,
        'TN_cumulative' : TN_cumulative,
        'cumulative_PPV': TP_cumulative / (TP_cumulative + FP_cumulative) * 100,
        'cumulative_NPV': TN_cumulative / (TN_cumulative + FN_cumulative) * 100
    }

axs[0].plot(df_sorted['overcapacity_abs'], df_sorted['cumulative_samples'], drawstyle='steps-post', color='grey')

# log scale for y-axis
axs[0].set_yscale('log')
axs[0].set_ylim(10, len(df_test)*2)
axs[0].set_ylabel('number of samples')

axs[0].set_xscale('log')
axs[0].yaxis.grid(True, which='major')
# axs[0].yaxis.grid(True, which='minor')
#
# set grid lines every order of magnitude
axs[0].set_yticks([10, 100, 1000, 10000])
# set minot ticks without labels between the major ticks but without grid lines

feas_preds = [ 'LL', 'OC', 'CH'] #'OC_0.05', 'OC_0.1',

feas_dicts = {
    feas_pred : get_cumulative_PPV_NPV(oc_vals, df_test, feas_pred) for feas_pred in feas_preds
}


for feas_pred, feas_dict in feas_dicts.items():
    axs[1].plot(oc_vals, feas_dict['cumulative_PPV'], marker='o', markersize = 4,
            linestyle='-', color = set_rwth_color(colors[feas_pred]), zorder = zorder[feas_pred])
    axs[1].plot(oc_vals, feas_dict['cumulative_NPV'], marker='x', linestyle='--',
            color = set_rwth_color(colors[feas_pred]), zorder = zorder[feas_pred])

axs[1].yaxis.grid(True, which='major')

axs[1].yaxis.set_major_locator(MultipleLocator(2))
# axs[2].yaxis.set_major_locator(MultipleLocator(20))


axs[0].set_xticklabels([])

axs[1].set_ylabel('PPV and NPV in \\%', color='black') # to create space for the shared ylabel
# axs[2].set_ylabel('PPV and NPV in \\%', color='black') # to create space for the shared ylabel

for ax in axs:
    ax.set_xscale('log')
    ax.set_xlim(7e-2, 5e2)
    ax.xaxis.grid(True, which='major')

axs[1].set_ylim(-4.5, 90)
axs[1].set_xlabel(r'treshold of true absolute overcapacity abs$(\varepsilon^\mathrm{OC})$ in \%', fontsize=10)

VSPLIT = 90
# set custom y scale
scale_name : str = create_piecewise_linear_scale(
    vmin=-4.5,
    vsplit=VSPLIT,
    vmax=101,
    split_pos=0.5   # 50% of axis for first segment
)
axs[1].set_yscale(scale_name)
axs[1].set_yticks(np.concatenate([
        # coarse ticks
        np.arange(0, 81, 20),
        # fine ticks
        np.arange(90, 100.1, 2)
]))

ax.hlines(VSPLIT, 1e-4, 1e3, set_rwth_color('black', 75), '--', label='scale split')
axs[1].set_ylim(-4.5, 100.5)

# dashed black line for NPV, solid black line for PPV
for feas_pred in feas_preds:
    axs[1].plot([], [], color=set_rwth_color(colors[feas_pred]), linestyle='-', label=f'{labels[feas_pred]}', markersize = 4)
axs[1].plot([1, 2], [200, 200], color='black', linestyle='-', label='PPV', marker='o', markersize = 4)
axs[1].plot([1, 2], [200, 200], color='black', linestyle='--', label='NPV', marker='x')


axs[1].legend()

# add ylabel to middle of two subplots
# fig.text(0.02, 1/3, 'PPV and NPV in \\%', va='center', rotation='vertical', fontsize=10)


# Save plot:
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)
plt.savefig(
    os.path.join(data_out, "plots", "fig11b.pdf"), dpi=300, bbox_inches='tight')

In [ ]:
def plot_cm(cm, ax_labels, ax):
    total = cm.sum()
    percentages = cm / total * 100

    labels = np.array([
        [f'TP\n{cm[0, 0]}\n{percentages[0, 0]:.1f}\\%', f'FN\n{cm[0, 1]}\n{percentages[0, 1]:.1f}\\%'],
        [f'FP\n{cm[1, 0]}\n{percentages[1, 0]:.1f}\\%', f'TN\n{cm[1, 1]}\n{percentages[1, 1]:.1f}\\%'] #%
    ])

    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=cm.sum() * 0.6)
    # add lines between the cells
    ax.axhline(0.5, color='black', linewidth=1)
    ax.axvline(0.5, color='black', linewidth=1)
    # annotate with absolute and relative values
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, labels[i, j], ha='center', va='center', color= 'white' if cm[i, j] > cm.sum() * 0.6 / 2 else 'black', fontsize=FONTSIZE)
    ax.set_xticks(np.arange(cm.shape[1]))
    ax.set_yticks(np.arange(cm.shape[0]))
    ax.set_xticklabels(ax_labels, fontsize=FONTSIZE)
    ax.set_yticklabels(ax_labels, rotation=90, va='center', fontsize=FONTSIZE)


    return im

n_day = 50

data_out = get_data_out(DATA_IN, f"project_ID14_{n_day}.yaml")
df_test = pd.read_csv(os.path.join(data_out, 'op_var_100000_test_with_feasibility_predictions.csv'))

# Convert id to int and sort
df_test['id'] = df_test['id'].astype(int)
df_test = df_test.sort_values(by='id')

# Absolute true overcapacity in percentage points
df_test['overcapacity_abs'] = df_test['overcapacity'].abs() * 100

plt.close('all')

FONTSIZE = 7
oc_thresholds = [0.1, 1, 10, 100] # [0.1, 100] #
feas_preds = ['LL', 'OC', 'OC_0.05', 'OC_0.1', 'CH']

labels = {
    'LL': r'ENS',
    'OC': r'OC, $\varepsilon^\mathrm{OC}_\mathrm{min}=0$\%',
    'OC_0.05':r'OC, $\varepsilon^\mathrm{OC}_\mathrm{min}=5$\%',
    'OC_0.1': r'OC, $\varepsilon^\mathrm{OC}_\mathrm{min}=10$\%',
    'CH': 'CH',
    'BM': 'BM',
    'noFP': 'none'
}
fig, axs = plt.subplots(len(oc_thresholds), len(feas_preds), figsize=(6.842, 5.8), constrained_layout=True)

for i, oc_threshold in enumerate(oc_thresholds):
    # print(i, oc_threshold)
    for j, feas_pred in enumerate(feas_preds):
        # print(j, feas_pred)
        df_subset = df_test[df_test['overcapacity_abs'] <= oc_threshold]

        if feas_pred == 'LL':
            y_pred = binarize_lost_load(df_subset['lost_load_predict'])

        elif feas_pred == 'OC':
            y_pred = binarize_overcapacity(df_subset['overcapacity_predict'], oc_min= 0.0)

        elif feas_pred == 'OC_0.05':
            y_pred = binarize_overcapacity(df_subset['overcapacity_predict'], oc_min= 0.05)

        elif feas_pred == 'OC_0.1':
            y_pred = binarize_overcapacity(df_subset['overcapacity_predict'], oc_min= 0.1)

        elif feas_pred == 'CH':
            y_pred = df_subset['feasibility_convexhull']

        else:
            raise ValueError(f"Unkown feas_pred: '{feas_pred}'")

        cm = confusion_matrix(binarize_overcapacity(df_subset['overcapacity']), y_pred, labels=[1, 0])

        plot_cm(cm, ax_labels=['feasible', 'infeasible'], ax=axs[i, j])

        if i == 0:
            # axs[i, j].set_title(f"{labels[feas_pred]}, {oc_threshold}\\%")
            axs[i, j].set_title(f"{labels[feas_pred]}", fontsize=FONTSIZE+2)

        if j == 0:
            # axs[i, j].set_ylabel("true abs($\\varepsilon^\\mathrm{OC}$) $\\leq$"+ f" {oc_threshold}\\%" + "\nactual", fontsize=FONTSIZE)
            # axs[i, j].set_ylabel("abs($\\varepsilon^\\mathrm{OC}$)$\\leq$"+ f"{oc_threshold}\\%" + "\nactual", fontsize=FONTSIZE+0.5)
            axs[i, j].set_ylabel("$|\\varepsilon^\\mathrm{OC}|\\leq$"+ f"{oc_threshold}\\%", fontsize=FONTSIZE+0.5)
        # else:
        #     axs[i, j].set_ylabel('actual', fontsize=FONTSIZE+0.5)

        # axs[i, j].set_xlabel('predicted', fontsize=FONTSIZE+0.5)
    
# Save plot:
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)
plt.savefig(
    os.path.join(data_out, "plots", "fig05_confusion_matrices_over_oc.pdf"), dpi=300, bbox_inches='tight')
print(data_out)

## **Step I.5** Influence of training set size:

In [ ]:
train_set_sizies = [5000, 10000, 20000, 40000, 80000]
train_set_size_default = 80000

df_train_full = pd.read_csv(os.path.join(DATA_OUT, 'op_var_100000_train.csv'))

ts_scaling_names_af = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)
processes_af = get_process_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True, storage_or_production='both')

ts_scaling_names = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=False)
processes = get_process_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=False, storage_or_production='both')

n_epochs = CONFIG['surrogate_models']['ANN_4L']['N_epochs']

for train_set_size in train_set_sizies:
    df_short = df_train_full.iloc[:train_set_size].copy(deep=True)
    filename = f"op_var_{train_set_size}_train.csv"
    df_short.to_csv(os.path.join(DATA_OUT, filename), index=False)

    new_n_epochs = n_epochs * train_set_size_default / train_set_size
    print(f"{train_set_size}: {int(new_n_epochs)}")


In [ ]:
# EXECUTE BEFORE PLOTTING
n_days = [5, 10, 50, 100]
train_set_sizies = [5000, 10000, 20000, 40000, 80000]

NN_names = {
    5000 : '5k',
    10000 : '10k',
    20000 : '20k',
    40000 : '40k',
    80000 : '80k',
}

ts_scaling_names_af = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True)
processes_af = get_process_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=True, storage_or_production='both')

ts_scaling_names = get_ts_scaling_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=False)
processes = get_process_names(DATA_IN, PROJECT_FILE, only_feasibility_affecting=False, storage_or_production='both')

In [ ]:

def calc_r2(y_true, y_pred):
    ss_res = (y_true - y_pred) ** 2
    ss_tot = (y_true - np.mean(y_true)) ** 2
    return 1 - (ss_res / ss_tot)

def calc_mse(y_true, y_pred):
    """Calculate Mean Squared Error."""
    return (y_true - y_pred) ** 2

def calc_mae(y_true, y_pred):
    "Calculate mean absolute error"
    return np.abs((y_true - y_pred))

def calc_npv_and_ppv(sm, df_test, feas_pred:str):

    
    if feas_pred == 'ENS':
    
        df_test['lost_load_predict'] = sm.predict(df_test)
        pred_labels = binarize_lost_load(df_test['lost_load_predict'])
        
        # Compute confusion matrix
        cm = confusion_matrix(binarize_lost_load(df_test['lost_load']), pred_labels, labels=[1, 0])
    elif feas_pred == 'OC':
        df_test['overcapacity_predict'] = sm.predict(df_test)
        pred_labels = binarize_overcapacity(df_test['overcapacity_predict']*100, oc_min=0)
        
        # Compute confusion matrix
        cm = confusion_matrix(binarize_overcapacity(df_test['overcapacity']*100), pred_labels, labels=[1, 0])
    else:
        raise ValueError(feas_pred)

    #ppend false positives and negatives
    TF = cm[0, 0] / len(df_test) *100  # Predicted feasible and actually feasible
    TI = cm[1, 1]/ len(df_test) *100  # Predicted infeasible and actually infeasible
    FF = cm[1, 0] / len(df_test) *100  # Predicted feasible but actually not feasible 
    FI = cm[0, 1] / len(df_test) * 100 # Predicted not feasible but actually feasible

    FOR = FI / (TI + FI) * 100 # False Omission rate
    NPV = 100 - FOR

    if (FF == 0) and (TF == 0):
        
        FDR = 100
        PPV = np.nan

    else:
        FDR = FF / (TF + FF) * 100 # False Discovery rate.
        PPV = 100 - FDR

    return NPV, PPV


mse_sm_opex = {}
mse_sm_oc = {}
mse_sm_ll = {}
mae_sm_opex = {}
mae_sm_oc = {}
mae_sm_ll = {}
r2_sm_opex = {}
r2_sm_oc = {}
r2_sm_ll = {}
npv_ll = {}
ppv_ll = {}
npv_oc = {}
ppv_oc = {}

sm_opex_dict = {n_day: {} for n_day in n_days}
sm_oc_dict = {n_day: {} for n_day in n_days}
sm_ll_dict = {n_day: {} for n_day in n_days}

for n_day in n_days:
    project_file =  f'project_ID14_{n_day}.yaml'
    data_out = get_data_out(DATA_IN, project_file)
    test_set = pd.read_csv(os.path.join(data_out, 'op_var_100000_test.csv'))

    for train_set_size in train_set_sizies:
        sm_opex = SurrogateModelNN(
            DATA_IN,
            project_file, 
            input_names =  ts_scaling_names + processes,
            output_name='OPEX_woLL',
            sm_name='ANN_4L')
        sm_opex.setup_sm()
        sm_opex.load_model(os.path.join(data_out, f'NN_{NN_names[train_set_size]}_opex'), f'NN_{NN_names[train_set_size]}_opex')
        sm_opex_dict[n_day][train_set_size] = sm_opex

        sm_oc = SurrogateModelNN(
            DATA_IN,
            project_file, 
            input_names =  ts_scaling_names_af + processes_af,
            output_name='overcapacity',
            sm_name='ANN_4L')
        sm_oc.setup_sm()
        sm_oc.load_model(os.path.join(data_out, f'NN_{NN_names[train_set_size]}_overcapacity'), f'NN_{NN_names[train_set_size]}_overcapacity')
        sm_oc_dict[n_day][train_set_size] = sm_oc

        sm_ll = SurrogateModelNN(
            DATA_IN,
            project_file, 
            input_names =  ts_scaling_names_af + processes_af,
            output_name='lost_load',
            sm_name='ANN_4L')
        sm_ll.setup_sm()
        sm_ll.load_model(os.path.join(data_out, f'NN_{NN_names[train_set_size]}_lostload'), f'NN_{NN_names[train_set_size]}_lostload')
        sm_ll_dict[n_day][train_set_size] = sm_ll


opex_max = max([
    max([
        sm_opex_dict[n_day][t_size].output_max
        for t_size in train_set_sizies
        ])
    for n_day in n_days 
    ])
oc_max = max([
    max([
        sm_oc_dict[n_day][t_size].output_max
        for t_size in train_set_sizies
        ])
    for n_day in n_days 
    ])
ll_max =  max([
    max([
        sm_ll_dict[n_day][t_size].output_max
        for t_size in train_set_sizies
        ])
    for n_day in n_days 
    ])
print(f"max(opex) = {opex_max:.2f} = {opex_max:.3e}")
print(f"max(oc)   = {oc_max:.2f} = {oc_max:.3e}")
print(f"max(ll)   = {ll_max:.2f} = {ll_max:.3e}")

for n_day in n_days:
    project_file =  f'project_ID14_{n_day}.yaml'
    data_out = get_data_out(DATA_IN, project_file)
    test_set = pd.read_csv(os.path.join(data_out, 'op_var_100000_test.csv'))

    # CALC MSE, SMAE and R2
    mse_sm_opex[n_day] = np.array([
        calc_mse(
            test_set['OPEX_woLL'].values / opex_max,
            sm_opex.predict(test_set).flatten() / opex_max
            )
        for sm_opex in sm_opex_dict[n_day].values()
    ])

    mse_sm_oc[n_day] = np.array([
        calc_mse(
            test_set['overcapacity'].values / oc_max,
            sm_oc.predict(test_set).flatten() / oc_max
            )
        for sm_oc in sm_oc_dict[n_day].values()
    ])

    mse_sm_ll[n_day] = np.array([
        calc_mse(
            test_set['lost_load'].values / ll_max,
            sm_ll.predict(test_set).flatten() / ll_max
            )
        for sm_ll in sm_ll_dict[n_day].values()
    ])

    r2_sm_opex[n_day] = [
        r2_score(test_set['OPEX_woLL'], sm_opex.predict(test_set))
        for sm_opex in sm_opex_dict[n_day].values()
    ]
    r2_sm_oc[n_day] = [
        r2_score(test_set['overcapacity'], sm_oc.predict(test_set))
        for sm_oc in sm_oc_dict[n_day].values()
    ]
    r2_sm_ll[n_day] = [
        r2_score(test_set['lost_load'], sm_ll.predict(test_set))
        for sm_ll in sm_ll_dict[n_day].values()
    ]

    mae_sm_opex[n_day] = np.array([
        calc_mae(
            test_set['OPEX_woLL'].values / opex_max,
            sm_opex.predict(test_set).flatten() / opex_max
            )
        for sm_opex in sm_opex_dict[n_day].values()
    ]).mean(axis=1)

    mae_sm_oc[n_day] = np.array([
        calc_mae(
            test_set['overcapacity'].values / oc_max,
            sm_oc.predict(test_set).flatten() / oc_max
            )
        for sm_oc in sm_oc_dict[n_day].values()
    ]).mean(axis=1)

    mae_sm_ll[n_day] = np.array([
        calc_mae(
            test_set['lost_load'].values / ll_max,
            sm_ll.predict(test_set).flatten() / ll_max
            )
        for sm_ll in sm_ll_dict[n_day].values()
    ]).mean(axis=1)


    npv_ppv_ens = [
        calc_npv_and_ppv(sm_ll, test_set, 'ENS')
        for sm_ll in sm_ll_dict[n_day].values() 
    ]

    npv_ll[n_day] = [val[0] for val in npv_ppv_ens]
    ppv_ll[n_day] = [val[1] for val in npv_ppv_ens]

    npv_ppv_oc = [
        calc_npv_and_ppv(sm_oc, test_set, 'OC')
        for sm_oc in sm_oc_dict[n_day].values() 
    ]

    npv_oc[n_day] = [val[0] for val in npv_ppv_oc]
    ppv_oc[n_day] = [val[1] for val in npv_ppv_oc]

In [ ]:
import pickle
# export as .yaml
export_dict = {
    'mse_sm_opex'   : mse_sm_opex,
    'mse_sm_oc'     : mse_sm_oc,
    'mse_sm_ll'     : mse_sm_ll,
    'mae_sm_opex'   : mae_sm_opex,
    'mae_sm_oc'     : mae_sm_oc,
    'mae_sm_ll'     : mae_sm_ll,
    'r2_sm_opex'    : r2_sm_opex,
    'r2_sm_oc'      : r2_sm_oc,
    'r2_sm_ll'      : r2_sm_ll,
    'npv_ll'        : npv_ll,
    'ppv_ll'        : ppv_ll,
    'npv_oc'        : npv_oc,
    'ppv_oc'        : ppv_oc,
    'opex_max'      : opex_max,
    'oc_max'        : oc_max,
    'll_max'        : ll_max,
    }
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
with open(os.path.join(data_out, "export_classifications.pkl"), 'wb') as f:
    pickle.dump(export_dict, f)

In [ ]:
# load from pickle:

import pickle
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
with open(os.path.join(data_out, "export_classifications.pkl"), 'rb') as f:
    export_dict = pickle.load(f)

mse_sm_opex = export_dict['mse_sm_opex']
mse_sm_oc   = export_dict['mse_sm_oc']
mse_sm_ll   = export_dict['mse_sm_ll']
mae_sm_opex = export_dict['mae_sm_opex']
mae_sm_oc   = export_dict['mae_sm_oc']
mae_sm_ll   = export_dict['mae_sm_ll']
r2_sm_opex  = export_dict['r2_sm_opex']
r2_sm_oc    = export_dict['r2_sm_oc']
r2_sm_ll    = export_dict['r2_sm_ll']
npv_ll      = export_dict['npv_ll']
ppv_ll      = export_dict['ppv_ll']
npv_oc      = export_dict['npv_oc']
ppv_oc      = export_dict['ppv_oc']
opex_max    = export_dict['opex_max']
oc_max      = export_dict['oc_max']
ll_max      = export_dict['ll_max']

In [ ]:


MARKERSIZE = 6
plt.close('all')
N_DAYS = 100
FIG_HEIGHT = 2

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

fig_main, ax_main = plt.subplots(1, 1, figsize=(3.296726719, FIG_HEIGHT), constrained_layout=True)
fig_SI, axs_SI = plt.subplots(1, 2, figsize=(6.842519441, FIG_HEIGHT), constrained_layout=True)

for ax in [ax_main, axs_SI[0]]:
    ax.plot(range(len(train_set_sizies)), r2_sm_opex[N_DAYS],  label='cost predictor',
            color=set_rwth_color('black', 100), marker='o',markersize=MARKERSIZE)
    ax.plot(range(len(train_set_sizies)), r2_sm_ll[N_DAYS], label='ENS predictor',
            color=set_rwth_color('orange', 100), marker='s',markersize=MARKERSIZE)
    ax.plot(range(len(train_set_sizies)), r2_sm_oc[N_DAYS], label='OC predictor',
            color=set_rwth_color('blue', 100), marker='v',markersize=MARKERSIZE)

    ax.set_ylabel("$R^2$")
    # ax.set_xlabel("combined training and validation set size")
    ax.set_xlabel("training set size")

    ax.set_xticks(range(len(train_set_sizies)))
    ax.set_xticklabels([int(0.9*val) for val in train_set_sizies])
    ax.grid()
    ax.legend(loc='best', framealpha=1)
    ax.yaxis.set_minor_locator(MultipleLocator(0.001))
    ax.yaxis.grid(True, which='major')
    ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)

    ax.set_ylim(0.995, 1.0003)

    # ax.annotate(
    #     f"{r2_sm_opex[0]:.3}",
    #     xy=(0.1, 0.995),          # arrow tip (data coordinates)
    #     xytext=(0, 0.996),     # text position (above it)
    #     ha="left",
    #     va="bottom",
    #     arrowprops=dict(
    #         arrowstyle="->",
    #         lw=1.2,
    #         color="black"
    #     )
    # )

    ytext = 0.99625
    xtest = 0.7

    yarrow = 0.995
    xarrow = xtest - ( ytext - yarrow ) / (r2_sm_opex[N_DAYS][1] - r2_sm_opex[N_DAYS][0]) * (1 - 0)
    ax.annotate(
        f"{r2_sm_opex[N_DAYS][0]:.3}",
        xy=(xarrow, yarrow),          # arrow tip (data coordinates)
        xytext=(xtest, ytext),     # text position (above it)
        ha="right",
        va="bottom",
        arrowprops=dict(
            arrowstyle="->",
            lw=1.2,
            color="black",
            relpos=(1.0, 0.0)
        )
    )

# MSE 
ax = axs_SI[1]
ax.plot(range(len(train_set_sizies)), np.sqrt(mse_sm_opex[N_DAYS].mean(axis=1)), label='cost predictor',
        color=set_rwth_color('black', 100), marker='o',markersize=MARKERSIZE)
ax.plot(range(len(train_set_sizies)), np.sqrt(mse_sm_ll[N_DAYS].mean(axis=1)), label='ENS predictor',
        color=set_rwth_color('orange', 100), marker='s', markersize=MARKERSIZE)
ax.plot(range(len(train_set_sizies)), np.sqrt(mse_sm_oc[N_DAYS].mean(axis=1)), label='OC predictor',
        color=set_rwth_color('blue', 100), marker='v',markersize=MARKERSIZE)

ax.set_ylabel("root mean square error")
ax.set_xlabel("combined training and validation set size")

ax.set_xticks(range(len(train_set_sizies)))
ax.set_ylim(0, ax.get_ylim()[1])

ax.set_xticklabels(list(train_set_sizies))
ax.grid(which='both')
ax.legend(loc='best')

# Save plots:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

fig_main.savefig(
    os.path.join(data_out, "plots", "fig05_osm_train_size_d100_MAIN.pdf"), dpi=300, bbox_inches='tight')


## **Step I.6** Influence of training set size for all # typical days on PPV and NPV of ENS pred

In [ ]:
# load from pickle:

import pickle
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
with open(os.path.join(data_out, "export_classifications.pkl"), 'rb') as f:
    export_dict = pickle.load(f)

mse_sm_opex = export_dict['mse_sm_opex']
mse_sm_oc   = export_dict['mse_sm_oc']
mse_sm_ll   = export_dict['mse_sm_ll']
mae_sm_opex = export_dict['mae_sm_opex']
mae_sm_oc   = export_dict['mae_sm_oc']
mae_sm_ll   = export_dict['mae_sm_ll']
r2_sm_opex  = export_dict['r2_sm_opex']
r2_sm_oc    = export_dict['r2_sm_oc']
r2_sm_ll    = export_dict['r2_sm_ll']
npv_ll      = export_dict['npv_ll']
ppv_ll      = export_dict['ppv_ll']
npv_oc      = export_dict['npv_oc']
ppv_oc      = export_dict['ppv_oc']
opex_max    = export_dict['opex_max']
oc_max      = export_dict['oc_max']
ll_max      = export_dict['ll_max']

In [ ]:

MARKERSIZE = 36

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

# Plotting the results
plt.close('all')
fig, axs = plt.subplots(2,2, figsize=(3.296726719, 3), sharex=True, sharey=True, constrained_layout=True)

YMIN = 90
YMAX = 100.5


for j, n_day in enumerate(n_days):
    ax = axs.flatten()[j]
    for i, train_set_size in enumerate(train_set_sizies):
        text = f"{(0.9*train_set_size/1000):.1f}" if train_set_size == 5000 else f"{int(0.9*train_set_size/1000)}"

        if np.isnan(ppv_ll[n_day][i]):
            ax.vlines([npv_ll[n_day][i]], ymin=YMIN, ymax=YMAX, color='k', linestyle = '--', linewidth=0.7)
            # ax.scatter(npv_ll[n_day][i], (YMAX + YMIN)/2, marker = 'o', color='k')

            ax.text(
                npv_ll[n_day][i] + 1,
                YMIN + (YMAX - YMIN) * 0.25 + i * (YMAX - YMIN) * 0.5 / len(train_set_sizies),
               text,
                ha='left', va='center')
        else:
            ax.scatter(npv_ll[n_day][i], ppv_ll[n_day][i], marker = 'o', color='k', s = np.sqrt(MARKERSIZE), zorder=5)
            # if n_day == 50 and train_set_size == 10000:
            #     x_offset = 2.5
            # else:
            #     x_offset = 0

            ax.text(
                npv_ll[n_day][i], # + x_offset,
                ppv_ll[n_day][i] - 0.2,
                text, #f"{(0.9*train_set_size/1000):.1f}",
                ha='right', va='top')

    ax.set_ylim([YMIN, YMAX])
    ax.set_xlim([35, 100])
    ax.grid()
    ax.set_title(f"{n_day} days")
    # ax.set_xlabel('NPV')
    
    ax.yaxis.grid(True, which='major')
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_minor_locator(MultipleLocator(1))
    ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
    
    ax.xaxis.grid(True, which='major')
    ax.xaxis.set_major_locator(MultipleLocator(20))
    ax.xaxis.set_minor_locator(MultipleLocator(10))
    ax.xaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)

for ax in axs[1,:]:
    ax.set_xlabel(r'NPV in \%')

for ax in axs[:,0]:
    ax.set_ylabel(r'PPV in \%')


# Save plots:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

fig.savefig(
    os.path.join(data_out, "plots", "fig05_ens_classification.pdf"), dpi=300, bbox_inches='tight')

## **Step I.7** Influence of training set size on R2 and SMAE for all # typical days

In [ ]:

MARKERSIZE = 6
plt.close('all')

FIG_HEIGHT = 6.5#8

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

fig, axs = plt.subplots(len(n_days), 2, figsize=(6.842519441, FIG_HEIGHT), constrained_layout=True)

X_TICKLABELS = [int(0.9*val) for val in train_set_sizies]
for i, n_day in enumerate(n_days):
    ax = axs[i, 0]
    ax.plot(range(len(train_set_sizies)), r2_sm_opex[n_day],  label='cost predictor',
            color=set_rwth_color('black', 100), marker='o',markersize=MARKERSIZE)
    ax.plot(range(len(train_set_sizies)), r2_sm_ll[n_day], label='ENS predictor',
            color=set_rwth_color('orange', 100), marker='s',markersize=MARKERSIZE)
    ax.plot(range(len(train_set_sizies)), r2_sm_oc[n_day], label='OC predictor',
            color=set_rwth_color('blue', 100), marker='v',markersize=MARKERSIZE)

    ax.set_ylabel("$R^2$")
    if i == len(n_days) - 1:
        ax.set_xlabel("training set size")

    ax.set_xticks(range(len(train_set_sizies)))
    ax.set_xticklabels(X_TICKLABELS)
    ax.grid()
    # ax.legend(loc='best')
    ax.yaxis.set_minor_locator(MultipleLocator(0.001))
    ax.yaxis.grid(True, which='major')
    ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)

    ax.set_ylim(0.995, 1.0003)

 
    ax.set_title(f"{n_day} days")

    # MSE 
    ax = axs[i,1]
    ax.plot(range(len(train_set_sizies)), mae_sm_opex[n_day]*100, label='cost predictor',
            color=set_rwth_color('black', 100), marker='o',markersize=MARKERSIZE)
    ax.plot(range(len(train_set_sizies)), mae_sm_ll[n_day]*100, label='ENS predictor',
            color=set_rwth_color('orange', 100), marker='s', markersize=MARKERSIZE)
    ax.plot(range(len(train_set_sizies)), mae_sm_oc[n_day]*100, label='OC predictor',
            color=set_rwth_color('blue', 100), marker='v',markersize=MARKERSIZE)

    ax.set_ylabel(r"$\mathrm{SMAE}$ in \%")
    if i == len(n_days) - 1:
        ax.set_xlabel("training set size")

    ax.set_xticks(range(len(train_set_sizies)))
    ax.set_ylim(0, 0.8)

    ax.set_xticklabels(X_TICKLABELS)
    # ax.grid(which='both')
    if i == 0:
        ax.legend(loc='best', framealpha=1)
    ax.set_title(f"{n_day} days")

    ax.yaxis.grid(True, which='major')
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.xaxis.grid(True, which='major')

# Save plots:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

fig.savefig(
    os.path.join(data_out, "plots", "fig05_osm_train_size_all_days.pdf"), dpi=300, bbox_inches='tight')

# **(II)** Use **Hybrid Surrogate Models (HSM)**

## **Step II.1** Generate Test Data for Design Optimization


The process of generating data can be found in demo_Sampling.ipynb in Section **1. Demonstration of sampling different timeseries**.

Gives: 
* ```design_opt_var_ts_lhs_sample_10800.csv```
* ```design_opt_var_ts_ts_ignore.yaml```

## **Step  II.2:** Calculate Test Data Outputs
Is Done on the HPC with ```script_JP1_design.sh```.

Gives:
* ```design_10800.csv``` (results of design optimization)

In [ ]:
# Or not on the HPC:
from sm.routines import design_optimization_var_ts, generate_aggregated_input_data
from core.helpers import UniqueKeyLoader

TS_IGNORE_FILE = 'design_opt_var_ts_ts_ignore.yaml'
CSV_INPUT_NAME = 'design_opt_var_ts_lhs_sample_10800.csv'

# Generate aggregated input data:
generate_aggregated_input_data(DATA_IN, PROJECT_FILE)

# Send timeseries_ignore to all workers
with open(os.path.join(DATA_COMPUTED_IN, TS_IGNORE_FILE), 'rb') as f:
    timeseries_ignore = yaml.load(f, Loader=UniqueKeyLoader)

df_in = pd.read_csv(os.path.join(DATA_COMPUTED_IN, CSV_INPUT_NAME))

# Design optimization
df_design = design_optimization_var_ts(
    df_in = df_in,
    data_in_path=DATA_IN,
    project_file=PROJECT_FILE,
    timeseries_ignored= timeseries_ignore,
    debug=False # True: Print solver log, for each optimization: export output_dict and result_plot
)
df_design.to_csv(os.path.join(DATA_OUT, 'design_10800.csv'), index=False)

If samples from ```design_10800.csv``` look weird (e.g., sample 103), they can be checked individually:

In [ ]:
from sm.routines import design_optimization_var_ts, generate_aggregated_input_data

generate_aggregated_input_data(DATA_IN, PROJECT_FILE)

df_design= pd.read_csv(os.path.join(DATA_OUT, 'design_10800.csv'))

df_check_design = design_optimization_var_ts(
    df_design.loc[[1]],
    DATA_IN,
    PROJECT_FILE,
    timeseries_ignored=None, # If timeseris are ignored, put them here
    debug=True # This is important for the debugging
)

## **Step  II.3**: Predict Test Data Outputs with Hybrid SM
Is done on the HPC with ```script_JP1_predict_hybrid_sm.sh```

Gives:
* ```design_hybrid_sm_lostload.csv``` (Predicted with lostload feasibility predictor)
* ```design_hybrid_sm_overcapacity.csv``` (Predicted with overcapacity feasibility predictor)

### **Step  II.3.1** Or not on the HPC (without routine)

In [ ]:
# --- Get names of the inputs for the surrogates:---
 
# Get ALL ts_scaling and process names:
ts_scaling_names = get_ts_scaling_names(
    DATA_IN, PROJECT_FILE,
    only_feasibility_affecting=False # <- Must be False
    )
processes = get_process_names(
    DATA_IN, PROJECT_FILE,
    only_feasibility_affecting=False, # <- Must be False
    storage_or_production='both'
    )

# Get ONLY FEASIBILITY AFFECTING ts_scaling and process names:
ts_scaling_names_af = get_ts_scaling_names(
    DATA_IN, PROJECT_FILE,
    only_feasibility_affecting=True # <- Must be True
    )
processes_af = get_process_names(
    DATA_IN, PROJECT_FILE,
    only_feasibility_affecting=True, # <- Must be True
    storage_or_production='both'
    )

In [ ]:
# --- Load trained Surrogate models:---
NAME_OPEX_SM = 'NN_opex'
NAME_LL_SM = 'NN_lostload'
NAME_OC_SM = 'NN_overcapacity'

### Load OPEX predictor:
sm_opex = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names = ts_scaling_names + processes,
    output_name='OPEX_woLL',
    sm_name='ANN_4L')

sm_opex.setup_sm()
# Load model:
sm_opex.load_model(model_path = os.path.join(DATA_OUT, NAME_OPEX_SM), filename = NAME_OPEX_SM)
# Or load specific parameters:
# sm_opex.sm.load_params(f_params=os.path.join(DATA_OUT, NAME_OPEX_SM,'params_397.pkl'))

### Load lostload predictor:
sm_ll = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names = ts_scaling_names_af + processes_af,
    output_name='lost_load',
    sm_name='ANN_4L')

sm_ll.setup_sm()
# Load model:
sm_ll.load_model(model_path = os.path.join(DATA_OUT, NAME_LL_SM), filename = NAME_LL_SM)
# # Or load specific parameters:
# sm_ll.sm.load_params(f_params=os.path.join(DATA_OUT, NAME_LL_SM,'params_226.pkl'))

### Load overcapacity predictor:
sm_oc = SurrogateModelNN(
    DATA_IN,
    PROJECT_FILE,
    input_names = ts_scaling_names_af + processes_af,
    output_name='overcapacity',
    sm_name='ANN_4L')

sm_oc.setup_sm()
# Load model:
sm_oc.load_model(model_path = os.path.join(DATA_OUT, NAME_OC_SM), filename = NAME_OC_SM)
# # Or load specific parameters:
# sm_oc.sm.load_params(f_params=os.path.join(DATA_OUT, NAME_OC_SM,'params_265.pkl'))


In [ ]:
# Build feasibility and cost predictors:
cost_pred_NN = CostPredictorNN(DATA_IN, PROJECT_FILE, sm_opex)

feas_pred_NN_oc = FeasibilityPredictorNN(
    DATA_IN, PROJECT_FILE, sm_oc, 'overcapacity',
    processes_full = cost_pred_NN.processes,
    timeseries_full= cost_pred_NN.timeseries
)

feas_pred_NN_ll = FeasibilityPredictorNN(
    DATA_IN, PROJECT_FILE, sm_ll, 'lost_load',
    processes_full = cost_pred_NN.processes,
    timeseries_full= cost_pred_NN.timeseries
)

feas_pred_noFP = FeasibilityPredictorNoFP(
    DATA_IN, PROJECT_FILE, None, 'noFP',
    processes_full = cost_pred_NN.processes,
    timeseries_full= cost_pred_NN.timeseries
)

In [ ]:
sado_oc = SADOMachineLearning(cost_pred_NN, feas_pred_NN_oc)

sado_ll = SADOMachineLearning(cost_pred_NN, feas_pred_NN_ll)

sado_noFP = SADOMachineLearning(cost_pred_NN, feas_pred_noFP)

In [ ]:
x = [2134.72470064, 62.29706561, 9632.90665262, 1500.22826447, 3590.80483857,  4974.04847675, 6361.74792662, 2269.23954752]

hthp_index = sado_oc.processes.index('HTHP')

# Double HTHP capacity (as nominal capacity in x refers to electricity, but we want to display heat)
x[hthp_index] = x[hthp_index] * 2

print(x)

In [ ]:
# --- Predict Optimal Designs using Hybrid Surrogate models:---
design_input_df = pd.read_csv(os.path.join(DATA_COMPUTED_IN, 'design_opt_var_ts_lhs_sample_10800.csv'))


result_hsm_oc = sado_oc.predict_optimal_design(
    design_input_df.loc[[1500]],
    options={'maxiter': 10000})#


res = result_hsm_oc['x']
print(f"TAC = {result_hsm_oc['fun']:.3e} EUR")
{processes[i]: cap for i, cap in enumerate(res)}

In [ ]:
# CONVEX HULL
vertex_df = pd.read_csv(os.path.join(DATA_OUT, "convex_hull_vertex_results_5000.csv"))

# Load surrogate model for feasibility:
feasiblity_lp = Feasibility_LP(DATA_IN, PROJECT_FILE, vertex_df)
# --- Setup hybrid surrogate model ---
sado_ch = SADOConexHull(cost_pred_NN, feasiblity_lp)

In [ ]:
# --- Predict Optimal Designs using Hybrid Surrogate models:---
design_input_df = pd.read_csv(os.path.join(DATA_COMPUTED_IN, 'design_opt_var_ts_lhs_sample_10800.csv'))

result_sado_ch = sado_ch.predict_optimal_design(
    design_input_df.loc[[1500]])#


res = result_sado_ch['x']
print(f"TAC = {result_sado_ch['fun']:.3e} EUR")
{processes[i]: cap for i, cap in enumerate(res)}

### **Step II.3.2** Or not on the HPC (with routine)

In [ ]:
from sm.routines import design_optimization_var_ts_hybrid_sm

design_input_df = pd.read_csv(os.path.join(DATA_COMPUTED_IN, 'design_opt_var_ts_lhs_sample_10800.csv'))

NAME_OPEX_SM = 'NN_OPEX'
NAME_LL_SM = 'NN_lostload'
NAME_OC_SM = 'NN_overcapacity'

# Predict using hybrid sm based on lost_lost sm:
df_design_hsm_ll = design_optimization_var_ts_hybrid_sm(
    df_in               = design_input_df.copy()[1000:1001],
    data_in_path        = DATA_IN,
    project_file        = PROJECT_FILE,
    opex_sm_name_config = 'ANN_4L',
    feas_sm_name_config = 'ANN_4L',
    opex_sm_name        = NAME_OPEX_SM,
    feas_sm_name        = NAME_LL_SM,
    feas_sm_type        = 'lost_load'
)
df_design_hsm_ll.to_csv(os.path.join(DATA_OUT, 'design_hybrid_sm_lostload.csv'), index=False)

# Predict using hybrid sm based on overcapacity sm:
df_design_hsm_oc = design_optimization_var_ts_hybrid_sm(
    df_in               = design_input_df.copy()[1000:1002],
    data_in_path        = DATA_IN,
    project_file        = PROJECT_FILE,
    opex_sm_name_config = 'ANN_4L',
    feas_sm_name_config = 'ANN_4L',
    opex_sm_name        = NAME_OPEX_SM,
    feas_sm_name        = NAME_OC_SM,
    feas_sm_type        = 'overcapacity'
)
df_design_hsm_oc.to_csv(os.path.join(DATA_OUT, 'design_hybrid_sm_overcapacity.csv'), index=False)

## **Step II.4**: Test Hybrid Surrogate Model Predictions

### **Step II.4.1**

In [ ]:
# Load HSM predictions:

df_hsm_ll = pd.read_csv(os.path.join(DATA_OUT, 'fix_grad', 'design_hybrid_sm_lostload_fix_grad.csv'))
df_hsm_oc= pd.read_csv(os.path.join(DATA_OUT, 'fix_grad', 'design_hybrid_sm_overcapacity_fix_grad.csv'))

df_hsm_oc_min0_05 = pd.read_csv(os.path.join(DATA_OUT, 'oc_min_5e-2', 'design_hybrid_sm_overcapacity_oc_min_5e-2.csv'))

cs_dict = {
    'LL': df_hsm_ll,
    'OC': df_hsm_oc,
    'OC 0.05': df_hsm_oc_min0_05,
}

In [ ]:
max_iter = 10000

def analyse_hsm_neg_vals(df):
    processes = get_process_names(DATA_IN, PROJECT_FILE, False, 'both')

    negative_value_dict = {}
    for process in processes:
        # add number of negative values per process
        negative_value_dict[process] = {
            'n': len(df[df[process] < 0]),
            'values': df[df[process] < 0][process].values
        }

    return negative_value_dict

plt.close('all')
fig, axs = plt.subplots(2,3, figsize = (18, 8))

ax = axs[0, 0]
for key, df in cs_dict.items():
    n_iter = df['n_iter']
    ax.boxplot(n_iter, positions=[list(cs_dict.keys()).index(key)], widths=0.5)

ax.set_xticks(range(len(cs_dict)))
ax.set_xticklabels(list(cs_dict.keys()))
ax.set_title('Number of Iterations per Case')

# in axs[0,1] plot number of iterations = max_iter
ax = axs[0, 1]
for key, df in cs_dict.items():
    n_iter = df['n_iter']
    ax.bar(list(cs_dict.keys()).index(key), len(n_iter[n_iter == max_iter]), width=0.5)
ax.set_xticks(range(len(cs_dict)))
ax.set_xticklabels(list(cs_dict.keys()))
ax.set_title(f'Number of Designs Reaching Max Iterations ({max_iter})')

# in axs[0,2] plot percentage of samples with constraint violation
ax = axs[0, 2]
for key, df in cs_dict.items():
    constraint_violations = df['constr_violation']
    n_constr_violation = len(constraint_violations[constraint_violations > 1E-6])
    percentage = n_constr_violation / len(constraint_violations) * 100
    ax.bar(list(cs_dict.keys()).index(key), percentage, width=0.5)
ax.set_xticks(range(len(cs_dict)))
ax.set_xticklabels(list(cs_dict.keys()))
ax.set_title('Percentage of Designs with Constraint Violation')

# in axs[1, 2] plot mean of constraint violation
ax = axs[1, 2]
for key, df in cs_dict.items():
    constraint_violations = df['constr_violation']
    mean_violation = np.mean(constraint_violations)
    ax.bar(list(cs_dict.keys()).index(key), mean_violation, width=0.5)
ax.set_xticks(range(len(cs_dict)))
ax.set_xticklabels(list(cs_dict.keys()))
ax.set_title('Mean Constraint Violation per Case')
ax.set_yscale('log')

# in axs[1,0] plot of computation time
ax = axs[1, 0]
for key, df in cs_dict.items():
    comp_time = df['execution_time']
    ax.boxplot(comp_time, positions=[list(cs_dict.keys()).index(key)], widths=0.5)
ax.set_xticks(range(len(cs_dict)))
ax.set_xticklabels(list(cs_dict.keys()))
ax.set_title('Computation Time per Case (in seconds)')
ax.set_ylabel('Computation Time (s)')

# in axs[1,1] plot number, mean and min of negative values in all processes
ax = axs[1, 1]
for key, df in cs_dict.items():
    negative_values = analyse_hsm_neg_vals(df)
    neg_vals =np.concatenate([info['values'] for info in negative_values.values()])
    ax.boxplot(neg_vals, positions=[list(cs_dict.keys()).index(key)], widths=0.5)
ax.set_xticks(range(len(cs_dict)))
ax.set_xticklabels(list(cs_dict.keys()))
ax.set_title('Negative Values in Processes per Case')

# set Grid in all subplots
for ax in axs.flatten():
    ax.grid(True)

# Print the negative values for each case
for key, df in cs_dict.items():
    neg_vals = analyse_hsm_neg_vals(df)
    print(f"\n{key}")

    # print total number of negative values and their mean and max
    total_neg_vals = np.concatenate([info['values'] for info in neg_vals.values()])
    if len(total_neg_vals) == 0:
        min_neg_val = '-'
        mean_neg_val = '-'
    else:
        min_neg_val = np.min(total_neg_vals)
        mean_neg_val = np.mean(total_neg_vals)

    print(f"\tTotal Negative Values Count: {len(total_neg_vals)}, "
          f"Min: {min_neg_val}, Mean: {mean_neg_val}")
    constraint_violations = df['constr_violation']
    print(f"\tConstraint Violations:\tmean: {np.mean(constraint_violations)}, max: {np.max(constraint_violations)}")

    # print number of designs with constraint violation
    n_constr_violation = len(constraint_violations[constraint_violations > 1E-6])
    print(f"\tNumber of Designs with Constraint Violation: {n_constr_violation} ({n_constr_violation/len(constraint_violations)*100:.2f}%)")

    # print computation time (mean)
    comp_time = df['execution_time']
    print(f"\tComputation Time (mean): {np.mean(comp_time)} s")

    # print average number of iterations and number of designs reaching max iterations
    n_iter = df['n_iter']
    print(f"\tAverage Number of Iterations: {np.mean(n_iter)}")
    print(f"\tNumber of Designs Reaching Max Iterations ({max_iter}): {len(n_iter[n_iter == max_iter])}")

### **Step II.4.2**: Export filtered HSM predictions

To check, how the designs, which are predicted using the HSMs actually would perform, operational
optimizations are carried out. These, however, require non-negative capacity values. The HSMs yield
negative capacity values in some cases (This is unwanted behaviour, this issue will hopefully be 
fixed in the future). Therefore, negative capacity values are set to zero, before the operational
optimizations are done.

In [ ]:
def export_hsm_predictions_for_operational_optimization(df, filename):

    # filter negative values for processes in df_hsm_ll_cache
    df_filtered = df.copy()
    processes = get_process_names(DATA_IN, PROJECT_FILE, False, 'both')

    # set all processes with negative values to 0
    for process in processes:
        df_filtered.loc[df_filtered[process] < 0, process] = 0

    # Remove column 'task_index': (avoids error when hpc tries to insert this column again:)
    if 'task_index' in df_filtered.columns:
        df_filtered = df_filtered.drop(['task_index'], axis=1)

    # Save dataframe:
    df_filtered.to_csv(os.path.join(DATA_COMPUTED_IN, filename), index=False)

export_hsm_predictions_for_operational_optimization(
    df_hsm_oc_constr,
    'design_hybrid_sm_overcapacity_constr_filtered.csv')

### **Step II.4.3**: Check HSM predictions with operational optimizations.

Can be done on the HPC with ```script_JP1_test_hybrid_sm_prediction.sh```.

Gives:
* ```design_hybrid_sm_lostload_cache_filtered_opex.csv```
* ```design_hybrid_sm_lostload_cache_filtered_max_oc.csv```
* ```design_hybrid_sm_overcapacity_cache_filtered_opex.csv```
* ```design_hybrid_sm_overcapacity_cache_filtered_max_oc.csv```

Can also be done with routines on other machine than HPC (not shown here.)

In [ ]:
# Load Data:

import numpy as np
import matplotlib.pyplot as plta
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.ticker import MultipleLocator
from core.helpers import set_rwth_color

# def combine_operational_and_max_overcapacity_results(
#         filename_operation, filename_maxoc, value_of_lostload, idx_min=0):
#     df_operation = pd.read_csv(filename_operation)[idx_min:]
#     df_operation = df_operation.sort_values(by='id', ascending=True)
#     # df_operation.reset_index(drop=True, inplace=True)

#     df_max_oc = pd.read_csv(filename_maxoc)[idx_min:]
#     df_max_oc = df_max_oc.sort_values(by='id', ascending=True)
#     # df_max_oc.reset_index(drop=True, inplace=True)

#     df_combined = df_operation.copy()
#     df_combined['overcapacity'] = df_max_oc['overcapacity']
#     df_combined['TAC_woLL'] = df_combined['TAC'] - df_combined['lost_load'] * value_of_lostload

#     # Check that both dataframes have the same length:
#     if len(df_operation) != len(df_max_oc):
#         print( "Warning: Incomplete resoluts" +
#             f"Dataframes in \n{filename_operation}\nand\n{filename_maxoc}\nhave different lengths:"
#             f" {len(df_operation)} != {len(df_max_oc)}"
#             )

#     # Check that both dataframes have the same ids:
#     elif not all(df_operation['id'].values == df_max_oc['id'].values):
#         raise ValueError(f"IDs in {filename_operation} and {filename_maxoc} do not match!")

#     return df_combined

IDX_MIN = 800

value_of_lostload = list(get_value_of_lost_load(DATA_IN, PROJECT_FILE).values())[0]

stats_dicts = {
    '5'        : {'days': 5,   'project': 'project_ID14_5.yaml',         'idx_min': 800},
    # '5_1e6'    : {'days': 5,   'project': 'project_ID11_5_1e6.yaml',     'idx_min': 800},
    # '5'   : {'days': 5,   'project': 'project_ID11_5_E10k.yaml',    'idx_min': 800},
    # '10'       : {'days': 10,  'project': 'project_ID11_10.yaml',        'idx_min': 800},
    '10'  : {'days': 10,  'project': 'project_ID14_10.yaml',   'idx_min': 800},
    # '50'       : {'days': 50,  'project': 'project_ID11_50.yaml',        'idx_min': 800},
    '50'  : {'days': 50,  'project': 'project_ID14_50.yaml',   'idx_min': 800},
    # '100'      : {'days': 100, 'project': 'project_ID11_100.yaml',       'idx_min': 800},
    '100' : {'days': 100, 'project': 'project_ID14_100.yaml', 'idx_min': 800},
}
hsm_pred_dict = {}
benchmark_dict = {}
sm_dict = {}

for key, stat_dict in stats_dicts.items():
    idx_min = stat_dict['idx_min']
    project = stat_dict['project']
    data_out = get_data_out(DATA_IN, project)


    hsm_pred_dict[f'LL_{key}'] = combine_operational_and_max_overcapacity_results(
        DATA_IN, project,
        filename_operation=os.path.join(data_out, 'design_hybrid_sm_lostload_filtered_opex.csv'),
        filename_maxoc =os.path.join(data_out,  'design_hybrid_sm_lostload_filtered_max_oc.csv'),
        idx_min=idx_min)
    hsm_pred_dict[f'OC_{key}'] = combine_operational_and_max_overcapacity_results(
        DATA_IN, project,
        filename_operation=os.path.join(data_out, 'design_hybrid_sm_overcapacity_filtered_opex.csv'),
        filename_maxoc =os.path.join(data_out,  'design_hybrid_sm_overcapacity_filtered_max_oc.csv'),
        idx_min=idx_min)
    hsm_pred_dict[f'OC_{key}_5e-2'] = combine_operational_and_max_overcapacity_results(
        DATA_IN, project,
        filename_operation=os.path.join(data_out, 'design_hybrid_sm_overcapacity_5e-2_filtered_opex.csv'),
        filename_maxoc =os.path.join(data_out,  'design_hybrid_sm_overcapacity_5e-2_filtered_max_oc.csv'),
        idx_min=idx_min)
    hsm_pred_dict[f'OC_{key}_1e-1'] = combine_operational_and_max_overcapacity_results(
        DATA_IN, project,
        filename_operation=os.path.join(data_out, 'design_hybrid_sm_overcapacity_1e-1_filtered_opex.csv'),
        filename_maxoc =os.path.join(data_out,  'design_hybrid_sm_overcapacity_1e-1_filtered_max_oc.csv'),
        idx_min=idx_min)
    hsm_pred_dict[f'CH_{key}'] = combine_operational_and_max_overcapacity_results(
        DATA_IN, project,
        filename_operation=os.path.join(data_out, 'design_hybrid_sm_convexhull_filtered_opex.csv'),
        filename_maxoc =os.path.join(data_out,  'design_hybrid_sm_convexhull_filtered_max_oc.csv'),
        idx_min=idx_min)
    hsm_pred_dict[f'noFP_{key}'] = combine_operational_and_max_overcapacity_results(
        DATA_IN, project,
        filename_operation=os.path.join(data_out, 'design_hybrid_sm_noFP_filtered_opex.csv'),
        filename_maxoc =os.path.join(data_out,  'design_hybrid_sm_noFP_filtered_max_oc.csv'),
        idx_min=idx_min)

    df_benchmark = pd.read_csv(os.path.join(data_out, 'design_10800.csv'))[idx_min:]
    
    benchmark_dict[f'LL_{key}']      = df_benchmark
    benchmark_dict[f'OC_{key}']      = df_benchmark
    benchmark_dict[f'OC_{key}_5e-2'] = df_benchmark
    benchmark_dict[f'OC_{key}_1e-1'] = df_benchmark
    benchmark_dict[f'CH_{key}']      = df_benchmark
    benchmark_dict[f'noFP_{key}']    = df_benchmark


# add TAC_IST to all dicts:
for key, df in hsm_pred_dict.items():
    # Assert same length:
    if len(benchmark_dict[key]) != len(df):
        raise ValueError(
            f"Length of benchmark and hsm prediction do not match for {key}"
            f" ({len(benchmark_dict[key])} != {len(df)})"
        )
    df['TAC_opt'] = benchmark_dict[key]['TAC']
    df['regret_wLL'] =    (df['TAC'].values - df['TAC_opt'].values)
    df['rel_regret_wLL'] =    (df['TAC'].values - df['TAC_opt'].values) / df['TAC_opt'].values * 100
    df['regret_woLL'] =    (df['TAC_woLL'].values - df['TAC_opt'].values)
    df['rel_regret_woLL'] =    (df['TAC_woLL'].values - df['TAC_opt'].values) / df['TAC_opt'].values * 100


for key, stat_dict in stats_dicts.items():
    sm_dict[key] = {
        'noFP'    : hsm_pred_dict[f'noFP_{key}'],
        'LL'      : hsm_pred_dict[f'LL_{key}'],
        'OC'      : hsm_pred_dict[f'OC_{key}'],
        'OC_0.05' : hsm_pred_dict[f'OC_{key}_5e-2'],
        'OC_0.1'  : hsm_pred_dict[f'OC_{key}_1e-1'],
        'CH'      : hsm_pred_dict[f'CH_{key}'] ,
    }

colors = {
    'LL': 'orange',
    'OC': 'blue',
    'OC_0.05': 'green',
    'OC_0.1': 'petrol',
    'CH': 'magenta',
    'BM': 'black',
    'noFP': 'red'
    }

labels = {
    'LL': r'ENS',
    'OC': r'OC, 0\%',
    'OC_0.05': r'OC, 5\%',
    'OC_0.1': r'OC, 10\%',
    'CH': 'CH',
    'BM': 'BM',
    'noFP': 'none'
}

#### **Step II.4.4a**: Investigate, influence of #days

In [ ]:
# Plot histogram of overcapacity:

FILTER_MIN = -0.15
FILTER_MAX = 0.2
plt.close('all')
fig, axs = plt.subplots(1, 2, figsize=(8,4), gridspec_kw={'width_ratios': [1, 0.8]}, constrained_layout=True)
N_DAYS = 10
N_BINS = 75

ax = axs[0]

ALPHA = 0.7
FACTOR = 100
candidates = colors.keys()

for candidate in candidates:
# only use values between -1 and 1;
    df_filtered = sm_dict[N_DAYS][candidate]['overcapacity'].copy()
    df_filtered = df_filtered[(df_filtered > FILTER_MIN) & (df_filtered < FILTER_MAX)]
    ax.hist(
        df_filtered.values * FACTOR,
        bins=N_BINS, color=set_rwth_color(colors[candidate], 100), alpha=ALPHA, label=candidate
    )


ax.set_xlim((FILTER_MIN*FACTOR, FILTER_MAX*FACTOR))

for candidate in candidates:
    ax.axvline(
        sm_dict[N_DAYS][candidate]['overcapacity'].mean()*FACTOR,
        color=set_rwth_color(colors[candidate], 100), linestyle='dashed', linewidth=1)
        # label=f'Mean {candidate}: {sm_dict[N_DAYS][candidate]['overcapacity'].mean():.4f}')

ax.set_xlabel("Overcapacity in %")
ax.set_ylabel("Frequency")
ax.set_title(f"Overcapacity, {N_DAYS} days")
ax.grid()

ax.legend(loc='best')

ax = axs[1]

share_of_infeasible = [
    float((sm_dict[N_DAYS][candidate]['overcapacity'] <= -1e-13).sum()) *100/ len(sm_dict[N_DAYS][candidate]['overcapacity'])
    for candidate in candidates
]
# plot bar chart
ax.bar(range(len(candidates)), share_of_infeasible, color=[set_rwth_color(colors[candidate], 100) for candidate in candidates], zorder=2,
       alpha=ALPHA)
for pos, candidate in enumerate(candidates):
    ax.text(pos, 30, f"{share_of_infeasible[pos]:.1f}%", ha='center', fontsize=8)

ax.set_ylabel("Share of infeasible samples in %")
ax.set_xticks(range(len(candidates)))
ax.set_xticklabels(candidates)
ax.grid(zorder=-1)
# plt.tight_layout()

In [ ]:
# Compare TAC vs. TAC_pred and feasibility vs feasibility pred
plt.close('all')
fig, axs = plt.subplots(1,4, figsize=(12,4))
df_plot = hsm_pred_dict['OC_10']

ax = axs[0]
# Plot scatter TAC vs. TAC_pred in ax1:
ax.plot(
    df_plot['TAC'], df_plot['TAC_pred'], marker='x',
    linestyle='', color='blue', label='TAC w/ LL'
)
ax.plot(
    df_plot['TAC_woLL'], df_plot['TAC_pred'], marker='x',
    linestyle='', color='red', label='TAC w/o LL'
)
ax.plot([0,ax.get_ylim()[1]],[0,ax.get_ylim()[1]], color='k',label='perfect fit')
ax.set_xlim(ax.get_ylim())
ax.legend(loc='best')

ax.set_xlabel("TAC IST")
ax.set_ylabel("TAC PRED")
ax.grid()
ax.set_title("predicted TAC of design vs actual TAC")

ax = axs[1]
# Plot scatter TAC_opt vs. TAC_pred in ax1:
ax.plot(
    df_plot['TAC_opt'], df_plot['TAC_pred'], marker='x',
    linestyle='', color='blue'
)
ax.plot([0,ax.get_ylim()[1]],[0,ax.get_ylim()[1]], color='k',label='perfect fit')
ax.set_xlim(ax.get_ylim())
ax.legend(loc='best')

ax.set_xlabel("TAC opt")
ax.set_ylabel("TAC PRED")
ax.grid()
ax.set_title("predicted TAC of design vs optimal TAC")

# Plot scatter Overcapacity vs. feasibility_pred in ax1:
ax = axs[2]
ax.plot(
    df_plot['overcapacity'],
    df_plot['feasibility_pred'], marker='x',
    linestyle='', color='blue', label='overcapacity'
)
# ax.set_xlim(ax.get_ylim())
ax.plot([0,ax.get_ylim()[1]],[0,ax.get_ylim()[1]], color='k',label='perfect fit')
ax.legend(loc='best')

ax.set_xlabel("overcapacity IST")
ax.set_ylabel("overcapacity PRED")
ax.grid()

# Plot scatter lostload vs. feasibility_pred in ax2:
ax = axs[3]
ax.plot(
    df_plot['lost_load'],
    df_plot['feasibility_pred'], marker='x',
    linestyle='', color='blue', label='lostload'
)
# ax.set_xlim(ax.get_ylim())
ax.plot([0,ax.get_ylim()[1]],[0,ax.get_ylim()[1]], color='k',label='perfect fit')
ax.legend(loc='best')

ax.set_xlabel("lost load IST")
ax.set_ylabel("lost load PRED")
ax.grid()
ax.set_title("predicted vs acutal lost load of design")

plt.tight_layout()

# print r2 score for TAC and feasibility
r2_tac = r2_score(
    df_plot['TAC'],
    df_plot['TAC_pred']
)
r2_tac_wo_ll = r2_score(
    df_plot['TAC_woLL'],
    df_plot['TAC_pred']
)
r2_overcapacity = r2_score(
    df_plot['overcapacity'],
    df_plot['feasibility_pred']
)
r2_lostload = r2_score(
    df_plot['lost_load'],
    df_plot['feasibility_pred']
)
print(f"R2 Score TAC: {r2_tac:.3f}")
print(f"R2 Score TAC w/o LL: {r2_tac_wo_ll:.3f}")
print(f"R2 Score Overcapacity: {r2_overcapacity:.3f}")
print(f"R2 Score Lostload: {r2_lostload:.3f}")


In [ ]:

# plot scatter of regret vs relative regret
plt.close('all')

# fig, axs = plt.subplots(5, 4, figsize=(12,10))
fig, axs = plt.subplots(4, 5, figsize=(12,9))

for i, (key, df) in enumerate(hsm_pred_dict.items()):
    ax = axs.flatten()[i]
# ax = axs[0]
    # filter: only small value for gas price:
    # df = df[df["pricebuy_electricity_mean"]<0.3].copy()
    ax.scatter(df['TAC_opt'], df['rel_regret_wLL'], s=0.5, label="w/ LL", zorder=1)

    dsartist = dsshow(
        df,
        ds.Point('TAC_opt', 'rel_regret_wLL'),
        ds.count(),
        vmin=0,
        vmax=10,
        norm="linear",
        aspect="auto",
        ax=ax,
        zorder=2
    )

    # plot horizontal line at 0
    ax.axhline(df['rel_regret_wLL'].mean(), color='k', linestyle='--', linewidth=1)
    ax.set_xlabel('TAC bm [EUR]')
    ax.set_ylabel('rel regret [%]')
    ax.set_ylim(-0.04 * ax.get_ylim()[1], ax.get_ylim()[1]*1.05)
    ax.set_xlim(-0.04 * ax.get_xlim()[1], ax.get_xlim()[1]*1.05)

    ax.set_title(key)
    ax.grid()

# ax.legend()
plt.tight_layout()

In [ ]:
# plot scatter of natural gas price vs relative regret
plt.close('all')

# fig, axs = plt.subplots(5, 4, figsize=(12,10))
fig, axs = plt.subplots(2, 4, figsize=(12,6))

for i, (key, df) in enumerate(hsm_pred_dict.items()):
    ax = axs.flatten()[i]
# ax = axs[0]
    # df = df[df["pricebuy_electricity_mean"]<0.3].copy()

    ax.scatter(df['pricebuy_gas_mean'], df['rel_regret_woLL'], s=2, label="w/ LL", zorder=1)

    dsartist = dsshow(
        df,
        ds.Point('pricebuy_gas_mean', 'rel_regret_woLL'),
        ds.count(),
        vmin=0,
        vmax=10,
        norm="linear",
        aspect="auto",
        ax=ax,
        zorder=2
    )

    # plot horizontal line at 0
    # ax.axhline(df['rel_regret_woLL'].mean(), color='k', linestyle='--', linewidth=1)
    ax.set_xlabel('gas price [EUR/kWh]')
    ax.set_ylabel('rel regret [%]')
    ax.set_ylim(-0.04 * ax.get_ylim()[1], ax.get_ylim()[1]*1.05)
    # ax.set_xlim(-0.04 * ax.get_xlim()[1], ax.get_xlim()[1]*1.05)
    ax.set_xlim(-0.04 * ax.get_xlim()[1], ax.get_xlim()[1]*1.05)

    ax.set_title(key)
    ax.grid()

# ax.legend()
plt.tight_layout()

In [ ]:
# plot correlation matrix
df_corr = df_oc_ID5.copy()
df_corr = df_corr.drop(columns=['id', 'termination_condition', 'n_iter', 'execution_time', 'overcapacity',
       'lost_load_electricity', 'lost_load_gas', 'lost_load_heat', 'task_index', 'message', 'success',
       'lost_load_cold'])
import seaborn as sns
plt.close('all')
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(
    df_corr.corr(),
    annot=True, fmt=".4f", cmap='coolwarm', ax=ax,
    vmin=-1, vmax=1, center=0, annot_kws={"size": 5}
)
ax.set_title('Correlation Matrix of CH 5k Design Variables')
plt.tight_layout()


In [ ]:
import matplotlib.transforms as mtransforms
from core.plotting_helpers import row_major_for_legend

# Plot Regret woLL for all days:
plt.close('all')

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

# Set up the plot
fig = plt.figure(figsize=(6.8425, 6), constrained_layout=True)#, gridspec_kw={'width_ratios': [3, 3]}) #[3, 1.4]
gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[0.06, 0.94], width_ratios=[3, 3])

ax_leg  = fig.add_subplot(gs[0, :])   # legend row spanning both columns
ax0     = fig.add_subplot(gs[1, 0])
ax1     = fig.add_subplot(gs[1, 1])
axs = [ax0, ax1]

n_members_per_group = len(list(sm_dict.values())[1])# - 1
n_groups = len(sm_dict)

group_positions = get_grouped_positions(n_groups, n_members_per_group, 0.7)
center_positions = get_center_positions(n_groups, n_members_per_group, 0.7)


for ax in axs:
    n_days = []
    for n_group, (key, subdict) in enumerate(reversed(sm_dict.items())):

        n_days.append(key)
        candidates =subdict.keys()# [name for name in subdict.keys() if name not in {'days', 'noFP'}]

        for n_member, candidate in enumerate(reversed(candidates)):
            df = subdict[candidate]
            data = (df['rel_regret_woLL'].values)
            pos = group_positions[n_group*n_members_per_group + n_member]
            p95 = plot_custom_violin(ax, data, pos,color=colors[candidate], s=36)

    # Plot group seperations
    trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
    ax.hlines(
        [(center_positions[i] + center_positions[i+1])/2 for i in range(len(center_positions) - 1)],
        xmin=0, xmax=1, color='grey', alpha=0.6, transform=trans)


legend_elements = [
    mpatches.Patch(
        facecolor=set_rwth_color(colors[candidate], 50),
        edgecolor=set_rwth_color(colors[candidate], 100), label=labels[candidate])
    for candidate in candidates
] + [
    Line2D([0], [0], color='k', lw=4, linestyle='-',
           label=r'25$^\mathrm{th}$--75$^\mathrm{th}$'),
    Line2D([0], [0], color='k', lw=2, linestyle='-',
           label=r'5$^\mathrm{th}$--95$^\mathrm{th}$'),
    Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='k', markerfacecolor='white', markersize=np.sqrt(50), label='mean'),
    Line2D([0], [0], color='k', lw=0, marker='^', markeredgecolor='k', markerfacecolor='white', markersize=np.sqrt(50), label='median'),
    Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='grey',
           markerfacecolor='grey', markersize=np.sqrt(1), label='outlier'),
]

# Axes on the right:
axs_r = []

for ax in axs:
    # Set ax on the left:
    ax.set_ylim(group_positions[0]-1, group_positions[-1]+1)
    ax.set_yticks(group_positions)

    # Set ax on the right:
    axs_r.append(ax.twinx())
    axs_r[-1].set_ylim(ax.get_ylim())
    axs_r[-1].set_yticks(center_positions)

    # Set xlabel:
    ax.set_xlabel(r"relative regret $\rho$ in \%")

    # Default: no yticklabels
    ax.set_yticklabels([])
    axs_r[-1].set_yticklabels([])

axs[0].set_yticklabels([labels[c] for c in candidates][::-1] * 4)
axs[0].set_ylabel("feasibility predictor")#, fontsize=10.5)

axs_r[1].set_yticklabels([f'{days} days' for days in n_days])
axs_r[1].set_ylabel("number of typical days")#, fontsize=10.5)

axs[0].xaxis.set_major_locator(MultipleLocator(1))
axs[0].xaxis.grid(True, which='major')
axs[0].xaxis.set_minor_locator(MultipleLocator(0.25))
# axs[0].xaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
axs[0].set_xlim(-1, 4)

axs[1].xaxis.set_major_locator(MultipleLocator(25))
axs[1].xaxis.grid(True, which='major')
axs[1].xaxis.set_minor_locator(MultipleLocator(5))
# axs[1].xaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
axs[1].set_xlim(-75, 100)

# But the grid line below the voilin patch
for ax in axs:
    ax.set_axisbelow(True)

# Legend
ax_leg.axis("off")
ncol = len(candidates)
ax_leg.legend(
    handles=row_major_for_legend(legend_elements, ncol),
    loc="upper center",
    ncol=ncol,
    frameon=True
)

# Save plot:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_violin_regret_woLL.pdf"), dpi=300)

In [ ]:
import matplotlib.transforms as mtransforms
from core.plotting_helpers import row_major_for_legend

def get_fraction_of_infeasible_samples(df, tol=1e-13):
    return len(df[df['overcapacity'] < -tol]) / len(df) * 100

def make_figure(panel_kinds, outpath):
    """
    panel_kinds: list like ['oc_full','oc_zoom','infeasible'] (SI)
                       or ['oc_zoom','infeasible'] (main)
    """
    plt.close('all')

    plt.rcParams['text.usetex'] = True
    plt.rcParams['font.family'] = 'serif'

    # --- Figure layout : choose grid based on panel count ---
    if len(panel_kinds) == 3:
        fig = plt.figure(figsize=(6.8425, 6), constrained_layout=True)
        gs = fig.add_gridspec(
            nrows=2, ncols=3,
            height_ratios=[0.06, 0.94],
            width_ratios=[2.5, 3, 2.5]
        )
        ax_leg = fig.add_subplot(gs[0, :])
        axs = [fig.add_subplot(gs[1, 0]),
               fig.add_subplot(gs[1, 1]),
               fig.add_subplot(gs[1, 2])]

    elif len(panel_kinds) == 2:
        fig = plt.figure(figsize=(6.8425, 6), constrained_layout=True)
        gs = fig.add_gridspec(
            nrows=2, ncols=2,
            height_ratios=[0.06, 0.94],
            width_ratios=[3, 2.5]   # zoom + infeasible
        )
        ax_leg = fig.add_subplot(gs[0, :])
        axs = [fig.add_subplot(gs[1, 0]),
               fig.add_subplot(gs[1, 1])]
    else:
        raise ValueError("panel_kinds must have length 2 or 3")

    n_members_per_group = len(list(sm_dict.values())[1])
    n_groups = len(sm_dict)

    group_positions = get_grouped_positions(n_groups, n_members_per_group, 0.7)
    center_positions = get_center_positions(n_groups, n_members_per_group, 0.7)

    # --- Main plotting loop: switch by panel_kinds[i] ---
    for i, ax in enumerate(axs):
        kind = panel_kinds[i]

        n_days = []
        for n_group, (key, subdict) in enumerate(reversed(sm_dict.items())):
            n_days.append(key)
            candidates = [name for name in subdict.keys() if name != 'days']

            for n_member, candidate in enumerate(reversed(candidates)):
                df = subdict[candidate]
                pos = group_positions[n_group*n_members_per_group + n_member]

                if kind in {'oc_full', 'oc_zoom'}:
                    data = df['overcapacity'].values * 100
                    print(f"{key} | {candidate}: {np.median(data):.2f} %")

                    p95 = plot_custom_violin(ax, data, pos, color=colors[candidate], s=36)

                    if len(panel_kinds) == 2 and data.mean() > 30:
                        X0 = 9
                        X1 = 19
                        ax.annotate(
                            "",
                            xy=(X1, pos), xytext=(X0,pos),
                            arrowprops=dict(arrowstyle="->", linewidth=1.1)
                        )
                                
                        # Place centered text above the arrow
                        ax.text(
                            (X0 + X1)/2,        # center in x
                            pos + 0.15,       # small offset above arrow
                            rf"mean at {data.mean():3.0f} \%",
                            ha="center",
                            va="bottom"
                        )
                elif kind == 'infeasible':
                    frac_infeasible = get_fraction_of_infeasible_samples(df)
                    ax.barh(
                        pos, frac_infeasible, height=0.7,
                        color=set_rwth_color(colors[candidate], 50),
                        edgecolor=set_rwth_color(colors[candidate], 100)
                    )
                    ax.text(
                        45, pos-0.135,
                        rf'{frac_infeasible:.2f}\%',
                        va='center', fontsize=8, ha='right',
                        color='black'
                    )

        # Plot group separations
        trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
        ax.hlines(
            [(center_positions[k] + center_positions[k+1]) / 2
             for k in range(len(center_positions) - 1)],
            xmin=0, xmax=1, color='grey', alpha=0.6, transform=trans
        )

    # --- Legend elements ---
    legend_elements = [
        mpatches.Patch(
            facecolor=set_rwth_color(colors[candidate], 50),
            edgecolor=set_rwth_color(colors[candidate], 100),
            label=labels[candidate]
        )
        for candidate in candidates
    ] + [
        Line2D([0], [0], color='k', lw=4, linestyle='-',
               label=r'25$^\mathrm{th}$--75$^\mathrm{th}$'),
        Line2D([0], [0], color='k', lw=2, linestyle='-',
               label=r'5$^\mathrm{th}$--95$^\mathrm{th}$'),
        Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='k',
               markerfacecolor='white', markersize=np.sqrt(50), label='mean'),
        Line2D([0], [0], color='k', lw=0, marker='^', markeredgecolor='k',
               markerfacecolor='white', markersize=np.sqrt(50), label='median'),
        Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='grey',
               markerfacecolor='grey', markersize=np.sqrt(1), label='outlier'),
    ]

    # --- Right axes (twinx), y ticks/labels ---
    axs_r = []
    for ax in axs:
        ax.set_ylim(group_positions[0]-1, group_positions[-1]+1)
        ax.set_yticks(group_positions)

        ax_r = ax.twinx()
        ax_r.set_ylim(ax.get_ylim())
        ax_r.set_yticks(center_positions)
        axs_r.append(ax_r)

        ax.set_yticklabels([])
        ax_r.set_yticklabels([])

    # Predictor labels go on first panel
    axs[0].set_yticklabels([labels[c] for c in candidates][::-1] * 4)
    axs[0].set_ylabel("feasibility predictor")

    # Typical-days labels go on last panel
    axs_r[-1].set_ylabel("number of typical days")
    axs_r[-1].set_yticklabels([f'{days} days' for days in n_days])

    # --- X labels / limits by panel kind  ---
    for ax, kind in zip(axs, panel_kinds):
        if kind == 'oc_full':
            ax.set_xlabel(r"overcapacity $\varepsilon^\mathrm{OC}$ in \%")
            ax.xaxis.set_major_locator(MultipleLocator(200))
            ax.xaxis.grid(True, which='major')
            ax.xaxis.set_minor_locator(MultipleLocator(50))
        elif kind == 'oc_zoom':
            ax.set_xlabel(r"overcapacity $\varepsilon^\mathrm{OC}$ in \%")
            ax.set_xlim(-10, 20)
            ax.xaxis.grid(True, which='major')
            ax.xaxis.set_minor_locator(MultipleLocator(1))
        elif kind == 'infeasible':
            ax.set_xlabel(r"infeasible samples in \%")
            ax.xaxis.grid(True, which='major')
            ax.xaxis.set_minor_locator(MultipleLocator(5))

        ax.set_axisbelow(True)

    # --- Legend axes ---
    ax_leg.axis('off')
    ncol = len(candidates)
    ax_leg.legend(
        handles=row_major_for_legend(legend_elements, ncol),
        loc='upper center',
        frameon=True,
        ncol=ncol
    )

    plt.savefig(outpath, dpi=300)

# -------------------------
# Build BOTH figures at once
# -------------------------
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

# Main text: drop full-range panel
# make_figure(
#     panel_kinds=['oc_zoom', 'infeasible'],
#     outpath=os.path.join(data_out, "plots", "fig05_violin_oc_MAIN.pdf")
# )

# SI: keep full-range panel
make_figure(
    panel_kinds=['oc_full', 'oc_zoom', 'infeasible'],
    outpath=os.path.join(data_out, "plots", "fig05_violin_oc_SI.pdf")
)


#### **Revision feas tol** 

In [ ]:
import matplotlib.transforms as mtransforms
from core.plotting_helpers import row_major_for_legend

def get_fraction_of_infeasible_samples(df, tol=1e-13):
    return len(df[df['overcapacity'] < -tol]) / len(df) * 100


plt.close('all')

feas_tols = [1e-15, 1e-13, 1e-6]
e_labels = ["$10^{-15}$", "$10^{-13}$", "$10^{-6}$"]

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'

fig = plt.figure(figsize=(6.8425, 6), constrained_layout=True)
gs = fig.add_gridspec(
    nrows=2, ncols=3,
    height_ratios=[0.06, 0.94],
    width_ratios=[1,1,1]
)
ax_leg = fig.add_subplot(gs[0, :])
axs = [fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[1, 2]),
    ]

n_members_per_group = len(list(sm_dict.values())[1])
n_groups = len(sm_dict)

group_positions = get_grouped_positions(n_groups, n_members_per_group, 0.7)
center_positions = get_center_positions(n_groups, n_members_per_group, 0.7)

# --- Main plotting loop: switch by panel_kinds[i] ---
for i, ax in enumerate(axs):

    # display title as 10^-13m ,,,
    ax.set_title(r"$\texttt{tol} = "  + rf"{feas_tols[i]:.0e}$")
    n_days = []
    for n_group, (key, subdict) in enumerate(reversed(sm_dict.items())):
        n_days.append(key)
        candidates = [name for name in subdict.keys() if name != 'days']

        for n_member, candidate in enumerate(reversed(candidates)):
            df = subdict[candidate]
            pos = group_positions[n_group*n_members_per_group + n_member]


            frac_infeasible = get_fraction_of_infeasible_samples(df, tol=feas_tols[i])
            ax.barh(
                pos, frac_infeasible, height=0.7,
                color=set_rwth_color(colors[candidate], 50),
                edgecolor=set_rwth_color(colors[candidate], 100)
            )
            ax.text(
                45, pos-0.135,
                rf'{frac_infeasible:.2f}\%',
                va='center', fontsize=8, ha='right',
                color='black'
            )

    # Plot group separations
    trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
    ax.hlines(
        [(center_positions[k] + center_positions[k+1]) / 2
        for k in range(len(center_positions) - 1)],
        xmin=0, xmax=1, color='grey', alpha=0.6, transform=trans
    )

# --- Legend elements ---
legend_elements = [
    mpatches.Patch(
        facecolor=set_rwth_color(colors[candidate], 50),
        edgecolor=set_rwth_color(colors[candidate], 100),
        label=labels[candidate]
    )
    for candidate in candidates
]
# --- Right axes (twinx), y ticks/labels ---
axs_r = []
for ax in axs:
    ax.set_ylim(group_positions[0]-1, group_positions[-1]+1)
    ax.set_yticks(group_positions)

    ax_r = ax.twinx()
    ax_r.set_ylim(ax.get_ylim())
    ax_r.set_yticks(center_positions)
    axs_r.append(ax_r)

    ax.set_yticklabels([])
    ax_r.set_yticklabels([])

# Predictor labels go on first panel
axs[0].set_yticklabels([labels[c] for c in candidates][::-1] * 4)
axs[0].set_ylabel("feasibility predictor")

# Typical-days labels go on last panel
axs_r[-1].set_ylabel("number of typical days")
axs_r[-1].set_yticklabels([f'{days} days' for days in n_days])

# --- X labels / limits by panel kind  ---
for ax in axs:
    ax.set_xlabel(r"infeasible samples in \%")
    ax.xaxis.grid(True, which='major')
    ax.xaxis.set_minor_locator(MultipleLocator(5))

    ax.set_axisbelow(True)

# --- Legend axes ---
ax_leg.axis('off')
ncol = len(candidates)
ax_leg.legend(
    handles=row_major_for_legend(legend_elements, ncol),
    loc='upper center',
    frameon=True,
    ncol=ncol
)

data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)
plt.savefig(os.path.join(data_out, "plots", "fig05_share_infeas_tol.pdf"), dpi=300)


#### **Step II.4.4b**: Investigate, where the high relative regret values come from

In [ ]:
from copy import deepcopy
# Do comparison for 50 typical days
N_DAYS = '100' # 50
sm_dict_N = deepcopy(sm_dict[N_DAYS])
FEAS_PRED = 'CH'
df_benchmark = benchmark_dict[f"{FEAS_PRED}_{N_DAYS}"]

process_name_dict = {
    'B': 'GB',
    'HTHP':'HTHP',
    'CHP': 'CHP',
    'TES': 'TES',
    'PV': 'PV',
    'CC': 'CC',
    'AC': 'AC',
    'battery': 'BAT'
}

# plt.rcParams['text.usetex'] = True  
# plt.rcParams['font.family'] = 'serif'
data_out = get_data_out(DATA_IN, f'project_ID14_{N_DAYS}.yaml')
PLOT_PATH = os.path.join(data_out, 'plots')
os.makedirs(PLOT_PATH, exist_ok=True)
from core.helpers import get_input_file, get_potential_capacity


# plot correlation matrix
df_combined = sm_dict_N[FEAS_PRED].copy()

df_corr = df_combined[['rel_regret_woLL'] + get_ts_scaling_names(DATA_IN, PROJECT_FILE, False)]
# add df_benchmark capacities:
processes = get_process_names(DATA_IN, PROJECT_FILE, False, 'both')
for process in processes:
    df_combined[f"{process}_bm"] = df_benchmark[process]
for process in processes:
    df_combined[f"{process}_diff"] = df_combined[process] - df_benchmark[process]
for process in processes:
    df_combined[f"{process}_absdiff"] = np.abs(df_combined[process] - df_benchmark[process])
    df_corr[f"{process}_absdiff"] = np.abs(df_combined[process] - df_benchmark[process])



technologies = get_input_file(DATA_IN, PROJECT_FILE, 'technologies')['technologies']
pot_cap = {process : get_potential_capacity(technologies[process]) for process in processes}

for process in processes:
    df_combined[f"{process}_norm"] = df_combined[process] / pot_cap[process]
    df_combined[f"{process}_bm_norm"] = df_combined[f"{process}_bm"] / pot_cap[process]
    df_combined[f"{process}_diff_norm"] = df_combined[f"{process}_diff"] / pot_cap[process]

In [ ]:
# plot scatter of regret vs relative regret
plt.close('all')

fig, axs = plt.subplots(2, 3, figsize=(12,7), constrained_layout=True)

for i, (key, df) in enumerate(sm_dict_N.items()):
    if key == 'days':
        continue

    ax = axs.flatten()[i]
    ax.scatter(df['TAC_opt'], df['rel_regret_wLL'], s=0.5, label="w/ LL", zorder=1)

    dsartist = dsshow(
        df,
        ds.Point('TAC_opt', 'rel_regret_wLL'),
        ds.count(),
        vmin=0,
        vmax=10,
        norm="linear",
        aspect="auto",
        ax=ax,
        zorder=2
    )

    # plot horizontal line at 0
    ax.axhline(df['rel_regret_wLL'].mean(), color='k', linestyle='--', linewidth=1)
    ax.set_xlabel('TAC bm [EUR]')
    ax.set_ylabel('rel regret [%]')
    ax.set_ylim(-0.04 * ax.get_ylim()[1], ax.get_ylim()[1]*1.05)
    ax.set_xlim(-0.04 * ax.get_xlim()[1], ax.get_xlim()[1]*1.05)

    ax.set_title(key)
    ax.grid()



In [ ]:

import seaborn as sns
plt.close('all')
# fig, ax = plt.subplots(figsize=(17, 12), constrained_layout=True)
fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)

# only lower triangle
sns.heatmap(
    df_corr.corr(),
    annot=True, fmt=".2f", cmap='coolwarm', ax=ax,
    vmin=-1, vmax=1, center=0, annot_kws={"size": 10}, # from 5
    mask=np.triu(np.ones_like(df_corr.corr(), dtype=bool))
)
ax.set_title(f'Correlation Matrix {FEAS_PRED}, {N_DAYS} typical days')

# save figure with high resolution
fig.savefig(os.path.join(PLOT_PATH, f'correlation_matrix_{FEAS_PRED}_{N_DAYS}days.png'), dpi=400)

In [ ]:
# plot predicted vs benchmark capacities:
plt.close('all')
fig, axs = plt.subplots(2, int(len(processes)/2), figsize=(12,5), constrained_layout=True)


for i, process in enumerate(processes):
    # plot scatter
    ax = axs.flatten()[i]
    
    dsartist = dsshow(
        df_combined,
        ds.Point(f"{process}_norm", f"{process}_bm_norm"),
        ds.count(),
        vmin=0,
        vmax=15,
        norm="linear",
        aspect="auto",
        ax=ax,
        zorder=2,
        width_scale=0.5,
        height_scale=0.5
    )

    ax.plot([0, 1], [0, 1], 'k', linestyle='--', linewidth=1)
    ax.set_ylim([-0.025, 1.025])
    ax.set_xlim([-0.025, 1.025])
    ax.set_xlabel("pred cap")
    ax.set_ylabel("bm cap")
    ax.set_title(process)

    ax.grid()

fig.suptitle(f"{FEAS_PRED}, {N_DAYS} typical days")

In [ ]:
# plot histogram of difference
plt.close('all')
fig, axs = plt.subplots(2, int(len(processes)/2), figsize=(12,4), constrained_layout=True)

for i, process in enumerate(processes):
    disp_name = process_name_dict[process]
    ax = axs.flatten()[i]
    ax.hist(df_combined[f"{process}_diff"]/pot_cap[process], bins=100, zorder=10, range=(-1,1),
            color=set_rwth_color(colors[FEAS_PRED]))

    pos = 0.1
    ypos_marker = pos * ax.get_ylim()[1] + (1 - pos) * ax.get_ylim()[0]

    mean = df_combined[f"{process}_diff"].mean()/pot_cap[process]
    ax.scatter([mean], [ypos_marker], marker='o', color='white', edgecolor='black', s=50, zorder=15, label='mean')
    ax.plot([mean, mean], [ypos_marker, 0], zorder=13, color='k', linewidth='1')

    median = df_combined[f"{process}_diff"].median()/pot_cap[process]
    ax.scatter([median], [ypos_marker], marker='^', color='white', edgecolor='black', s=50, zorder=15, label='median')
    ax.plot([median, median], [ypos_marker, 0], zorder=13, color='k', linewidth='1')

    ax.set_xlabel( fr'$(\delta_\mathrm{{{disp_name}}}^\mathrm{{SADO}}-\delta_\mathrm{{{disp_name}}}^*) / \delta_\mathrm{{{disp_name}}}^\mathrm{{pot}}$')
    ax.set_title(process_name_dict[process])
    if i == 0:
        ax.legend()
    ax.grid()
    ax.set_xlim([-1.05, 1.05])
    ax.set_ylim([-0.025 * ax.get_ylim()[1], ax.get_ylim()[1]])
    if i in {0, 4}:
        ax.set_ylabel("frequency")

fig.suptitle(f"Histogram of difference in design capacities for {FEAS_PRED}, {N_DAYS} typical days")
# Save figure in high resolution
fig.savefig(os.path.join(PLOT_PATH, f'hist_diff_pred_bm_normalized_{FEAS_PRED}.png'), dpi=300)

In [ ]:
# plot histogram of surrogate capacity
plt.close('all')
fig, axs = plt.subplots(2, int(len(processes)/2), figsize=(12,4), constrained_layout=True)

for i, process in enumerate(processes):
    disp_name = process_name_dict[process]
    ax = axs.flatten()[i]
    ax.hist(df_combined[process]/pot_cap[process], bins=70, zorder=20,
            color=set_rwth_color(colors[FEAS_PRED]))

    pos = 0.1
    ypos_marker = pos * ax.get_ylim()[1] + (1 - pos) * ax.get_ylim()[0]

    ax.set_xlabel( fr'$\delta_\mathrm{{{disp_name}}}^* / \delta_\mathrm{{{disp_name}}}^\mathrm{{pot}}$')
    ax.set_title(process_name_dict[process])
    # if i == 0:
    #     ax.legend()
    ax.grid()
    ax.set_xlim([-0.025, 1.025])
    ax.set_ylim([-0.025 * ax.get_ylim()[1], ax.get_ylim()[1]])
    if i in {0, 4}:
        ax.set_ylabel("frequency")

fig.suptitle(f"Histogram of benchmark design capacities for  {N_DAYS} typical days")
# Save figure in high resolution
fig.savefig(os.path.join(PLOT_PATH, f'hist_{FEAS_PRED}_cap_{N_DAYS}.png'), dpi=300)

In [ ]:
# plot histrogram of capacity for all feasibility predictors:

plt.close('all')
sm_dict_N_wBM = sm_dict_N
sm_dict_N_wBM['BM'] = df_benchmark
fig, axs = plt.subplots(len(processes), len(sm_dict_N_wBM), figsize=(10,11), constrained_layout=True)

for j, process in enumerate(processes):

    # collect ylims:
    ylim_max_list = []
    for i, (feas_pred, data_df) in enumerate(sm_dict_N_wBM.items()):

        ax = axs[j, i]

        disp_name = process_name_dict[process]

        ax.hist(data_df[process]/pot_cap[process], bins=60, zorder=20, range=(0,1),
                color=set_rwth_color(colors[feas_pred]))

        if feas_pred == 'BM':
            ax.set_xlabel( fr'$\delta_\mathrm{{{disp_name}}}^* / \delta_\mathrm{{{disp_name}}}^\mathrm{{pot}}$')
        else:
            ax.set_xlabel( fr'$\delta_\mathrm{{{disp_name}}}^\mathrm{{SADO}} / \delta_\mathrm{{{disp_name}}}^\mathrm{{pot}}$')
        
        ax.xaxis.set_minor_locator(MultipleLocator(0.25))
        ax.xaxis.set_major_locator(MultipleLocator(0.5))
        ax.xaxis.grid(True, which='minor')
        # axs[1].xaxis.grid(True, which='major')
        ax.grid()
        ax.set_xlim([-0.025, 1.025])
        ylim_max_list.append(ax.get_ylim()[1])
        # ax.set_ylim([-0.025 * ax.get_ylim()[1], ax.get_ylim()[1]])
        if i == 0:
            ax.set_ylabel(f"{disp_name}\nfrequency")
        else:
            ax.tick_params(labelleft=False)

        if j == 0:
            ax.set_title(feas_pred)
        if j != len(processes) - 1:
            ax.tick_params(labelbottom=False)

    # Set max y_lim
    ylim_max = max(ylim_max_list)
    for i in range(len(sm_dict_N)):
        axs[j,i].set_ylim([-0.025 * ylim_max, ylim_max])

# fig.savefig(os.path.join(PLOT_PATH, f'hist_all_feaspreds_cap_{N_DAYS}.png'), dpi=300)

In [ ]:
# plot histrogram of differnece for all feasibility predictors:

plt.close('all')
fig, axs = plt.subplots(len(processes), len(sm_dict_N), figsize=(10,11), constrained_layout=True)

for j, process in enumerate(processes):

    # collect ylims:
    ylim_max_list = []
    for i, (feas_pred, data_df) in enumerate(sm_dict_N.items()):

        ax = axs[j, i]

        disp_name = process_name_dict[process]

        ax.hist((data_df[process] - df_benchmark[process])/pot_cap[process], bins=60, zorder=20, range=(-1,1),
                color=set_rwth_color(colors[feas_pred]))

        ax.set_xlabel( fr'$(\delta_\mathrm{{{disp_name}}}^\mathrm{{SADO}}-\delta_\mathrm{{{disp_name}}}^*) / \delta_\mathrm{{{disp_name}}}^\mathrm{{pot}}$')
        
        ax.xaxis.set_minor_locator(MultipleLocator(0.5))
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.xaxis.grid(True, which='minor')
        # axs[1].xaxis.grid(True, which='major')
        ax.grid()
        ax.set_xlim([-1.025, 1.025])
        ylim_max_list.append(ax.get_ylim()[1])

        if i == 0:
            ax.set_ylabel(f"{disp_name}\nfrequency")
        else:
            ax.tick_params(labelleft=False)

        if j == 0:
            ax.set_title(feas_pred)
        if j != len(processes) - 1:
            ax.tick_params(labelbottom=False)

    # Set max y_lim
    ylim_max = max(ylim_max_list)
    for i in range(len(sm_dict_N)):
        axs[j,i].set_ylim([-0.025 * ylim_max, ylim_max])


# fig.savefig(os.path.join(PLOT_PATH, f'hist_diff_all_feaspreds_cap_{N_DAYS}.png'), dpi=300)


In [ ]:
# Plot voilins of normalized design deviation for all capacities and feas. preds.
plt.close('all')

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

# Set up the plot
fig= plt.figure(figsize=(get_latex_textwidth(), 7.5), constrained_layout=True)
gs = fig.add_gridspec(nrows=5, ncols=2, height_ratios=[0.2, 1, 1, 1, 1], width_ratios=[1, 1])

ax_leg  = fig.add_subplot(gs[0, :])   # legend row spanning both columns

axs = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[2, 0]),
    fig.add_subplot(gs[2, 1]),
    fig.add_subplot(gs[3, 0]),
    fig.add_subplot(gs[3, 1]),
    fig.add_subplot(gs[4, 0]),
    fig.add_subplot(gs[4, 1])
]
for j, (process, ax) in enumerate(zip(processes, axs)):

    # collect ylims:
    ylim_max_list = []
    for i, (feas_pred, data_df) in enumerate(reversed(sm_dict_N.items())):
        plot_custom_violin(
            ax,
            data = (data_df[process] - df_benchmark[process])/pot_cap[process],
            pos = i,
            color = colors[feas_pred],
        )
        # print(feas_pred)
        if feas_pred == 'CH':
            ax.scatter(
                (data_df[process][900] - df_benchmark[process][900]) / pot_cap[process], i, s=30, marker='x', zorder =5)

    ax.set_yticks(range(len(sm_dict_N)))
    if j % 2 == 0:
        ax.set_yticklabels( [labels[c] for c in reversed(sm_dict_N.keys())] )
        ax.set_ylabel("feasibility predictor")
    else:
        ax.set_yticklabels([])

    if j in {6,7}:
        ax.set_xlabel(
            fr'$(d_c^\mathrm{{SADO}}-d_c^*) / d_c^\mathrm{{pot}}$')


    disp_name = process_name_dict[process]
    ax.set_title(rf"$c = $ {disp_name}")
    ax.set_xlim(-1.05, 1.05)
    ax.grid()

# Legend
legend_elements = [
    mpatches.Patch(
        facecolor=set_rwth_color(colors[candidate], 50),
        edgecolor=set_rwth_color(colors[candidate], 100), label=labels[candidate])
    for candidate in sm_dict_N
] + [
    Line2D([0], [0], color='k', lw=4, linestyle='-',
           label=r'25$^\mathrm{th}$--75$^\mathrm{th}$'),
    Line2D([0], [0], color='k', lw=2, linestyle='-',
           label=r'5$^\mathrm{th}$--95$^\mathrm{th}$'),
    Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='k', markerfacecolor='white', markersize=np.sqrt(50), label='mean'),
    Line2D([0], [0], color='k', lw=0, marker='^', markeredgecolor='k', markerfacecolor='white', markersize=np.sqrt(50), label='median'),
    Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='grey',
           markerfacecolor='grey', markersize=np.sqrt(1), label='outlier'),
]
ax_leg.axis("off")
ncol = len(candidates)
ax_leg.legend(
    handles=row_major_for_legend(legend_elements, ncol),
    loc="upper center",
    ncol=ncol,
    frameon=True
)

# Save plot:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_deviation_design_SI.pdf"), dpi=300)

In [ ]:
# plot rel regret vs design deviation
import datashader as ds
from datashader.mpl_ext import dsshow
plt.close('all')

def using_datashader(ax, x, y):

    df = pd.DataFrame(dict(x=x, y=y))
    dsartist = dsshow(
        df,
        ds.Point("x", "y"),
        ds.count(),
        vmin=0,
        vmax=50,
        norm="linear",
        aspect="auto",
        ax=ax,
        width_scale = 1,
        height_scale = 1,
        zorder=5.1
    )

    return dsartist

fig, axs = plt.subplots(2,4, figsize = (get_latex_textwidth(), 3.2), constrained_layout=True)

for j, (process, ax) in enumerate(zip(processes, axs.flatten())):
    ax.scatter(
        (sm_dict_N['CH'][process] - df_benchmark[process]) / pot_cap[process],
        sm_dict_N['CH']['rel_regret_woLL'],  s=3, marker='.', zorder =5, color='#440154', rasterized=True)
    dsartist = using_datashader(
        ax,
        (sm_dict_N['CH'][process] - df_benchmark[process]) / pot_cap[process],
        sm_dict_N['CH']['rel_regret_woLL'])

    ax.set_title(rf"$c = $ { process_name_dict[process]}")
    ax.set_xlim(-1.05,1.05)
    ax.set_ylim(-0.5, 30)
    ax.grid()

    ax.xaxis.set_minor_locator(MultipleLocator(0.25))
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.grid(True, which='minor', linestyle='-', linewidth=0.15)

    ax.yaxis.set_minor_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MultipleLocator(20))
    ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.15)

    if j > 3:
        ax.set_xlabel(
            r'$(\hat{d}_c-d_c^*) / d_c^\mathrm{{pot}}$')
    if j % 4 == 0:
        ax.set_ylabel(r'relative regret $\rho$ in \%')

# global colorbar
cbar = fig.colorbar(
    dsartist,
    ax=axs,          # IMPORTANT: all subplots
    location='right',
    shrink=0.9,
    pad=0.02
)

cbar.set_label("number of samples")

ticks = cbar.get_ticks()
labels = [f"{int(t)}" for t in ticks]
labels[-1] = f"{labels[-1]}+"

cbar.set_ticks(ticks)
cbar.set_ticklabels(labels)

# Save plot:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_scatter_regret_vs_design-deviation.pdf"), dpi=300)

In [ ]:
# MAIN: Plot r2 of normalized design deviation for all capacities and feas. preds.
plt.close('all')

sm_dict_N = deepcopy(sm_dict['100'])

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

hatch_dict = {
   'noFP': r'\\',
   'LL' : '--',
   'OC' : 'oo',
   'OC_0.05' : '//',
   'OC_0.1': "..",
   'CH': ''
}
# Set up the plot
fig = plt.figure(figsize=(get_latex_textwidth(), 2.4), constrained_layout=True)
gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[0.10, 0.45]#, 0.45],
    # width_ratios=[2.5, 3, 2.5]
)

ax_leg = fig.add_subplot(gs[0])
ax = fig.add_subplot(gs[1])
color_list = [set_rwth_color(colors[i], 75) for i in sm_dict_N]
WIDTH = 0.87

n_members_per_group = len(sm_dict_N.items())
group_positions = np.array([-5/12, -3/12, -1/12, 1/12, 3/12, 5/12]) * WIDTH

for j, process in enumerate(processes):

    # collect ylims:
    ylim_max_list = []
    data_rmse = []
    data_r2 = []
    data_smae = []

    for i, (feas_pred, data_df) in enumerate(sm_dict_N.items()):
        data_r2.append(r2_score(y_true = df_benchmark[process], y_pred = data_df[process]))

    ax.bar(j + group_positions, data_r2, color=color_list, width=WIDTH/6,
           zorder = 2, edgecolor = 'k', hatch=[hatch_dict[f] for f in sm_dict_N])

    print(f"{process}: {data_r2}")
# R2
ax.yaxis.set_major_locator(MultipleLocator(0.5))
ax.yaxis.grid(True, which='major')
ax.yaxis.set_minor_locator(MultipleLocator(0.1))
ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
ax.set_ylabel("$R^2$")
ax.set_ylim(-0.5,1)

ax.set_xticks(range(len(processes)))
ax.set_xticklabels([ process_name_dict[process] for process in processes])

# Legend:
ax_leg.axis("off")
ncol = n_members_per_group
legend_elements = [
    mpatches.Patch(
        facecolor=set_rwth_color(colors[f], 75),
        edgecolor='k', label=labels[f], hatch = hatch_dict[f])
    for f in sm_dict_N
]

ax_leg.legend(
    handles= legend_elements,
    loc="upper center",
    ncol=ncol,
    frameon=True
)


data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_deviation_design_r2_MAIN.pdf"), dpi=300)

In [ ]:
# SI: Plot r2 of normalized design deviation for all capacities and all days
plt.close('all')

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

hatch_dict = {
   'noFP': r'\\',
   'LL' : '--',
   'OC' : 'oo',
   'OC_0.05' : '//',
   'OC_0.1': "..",
   'CH': ''
}
# Set up the plot
fig = plt.figure(figsize=(get_latex_textwidth(), 6.8), constrained_layout=True)
gs = fig.add_gridspec(
    nrows= 1+ len(sm_dict), ncols=2,
    height_ratios= [0.10] + [0.45] * len(sm_dict),
)

ax_leg   = fig.add_subplot(gs[0,:])
axs_r2   = [fig.add_subplot(gs[i+1,0]) for i in range(len(sm_dict))]
axs_smae = [fig.add_subplot(gs[i+1,1]) for i in range(len(sm_dict))]

color_list = [set_rwth_color(colors[i], 75) for i in list(sm_dict.values())[-1]]
WIDTH = 0.87

group_positions = np.array([-5/12, -3/12, -1/12, 1/12, 3/12, 5/12]) * WIDTH

for k, (n_days, sm_dict_N) in enumerate(sm_dict.items()):
                                        
    for j, process in enumerate(processes):

        # collect ylims:
        ylim_max_list = []
        data_rmse = []
        data_r2 = []
        data_smae = []

        for i, (feas_pred, data_df) in enumerate(sm_dict_N.items()):
            data_r2.append(r2_score(y_true = df_benchmark[process], y_pred = data_df[process]))
            data_rmse.append(
                np.sqrt(
                    calc_mse(
                        y_true = df_benchmark[process] / pot_cap[process],
                        y_pred = data_df[process] / pot_cap[process]).mean()
                )
            )
            data_smae.append(
                (
                    calc_mae(
                        y_true = df_benchmark[process],
                        y_pred = data_df[process]
                        ) / pot_cap[process]# * 100
    #                    ) / (df_benchmark[process].max() - df_benchmark[process].min())# * 100

                        ).mean()
            )


        axs_r2[k].bar(j + group_positions, data_r2, color=color_list, width=WIDTH/6,
            zorder = 3, edgecolor = 'k', hatch=[hatch_dict[f] for f in sm_dict_N])

        axs_smae[k].bar(j + group_positions, np.array(data_smae)*100, color=color_list, width=WIDTH/6,
            zorder = 2.3, edgecolor = 'k', hatch=[hatch_dict[f] for f in sm_dict_N])

    # R2
    ax = axs_r2[k]
    ax.yaxis.set_major_locator(MultipleLocator(0.5))
    ax.yaxis.grid(True, which='major')
    ax.yaxis.set_minor_locator(MultipleLocator(0.1))
    ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
    ax.set_title(rf"{n_days} typical days")
    ax.set_ylabel("$R^2$")
    ax.set_ylim(-0.05,1)
    ax.set_xticks(range(len(processes)))
    ax.set_xticklabels([ process_name_dict[process] for process in processes])

    # SMAE
    ax = axs_smae[k]
    ax.yaxis.set_major_locator(MultipleLocator(10))
    ax.yaxis.grid(True, which='major')
    ax.yaxis.set_minor_locator(MultipleLocator(2))
    ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
    # ax.set_title(rf"root mean square error RMSE")
    # ax.set_ylabel(r"RMSE in \%")
    # ax.set_title(rf"standardized mean absolute error SMAE")
    ax.set_title(rf"{n_days} typical days")
    ax.set_ylabel(r"$\mathrm{SMAE}$ in \%")
    ax.set_ylim(-45/20,45)
    ax.set_xticks(range(len(processes)))
    ax.set_xticklabels([ process_name_dict[process] for process in processes])

# Legend:
n_members_per_group = len(list(sm_dict.values())[-1].items())
ax_leg.axis("off")
ncol = n_members_per_group
legend_elements = [
    mpatches.Patch(
        facecolor=set_rwth_color(colors[f], 75),
        edgecolor='k', label=labels[f], hatch = hatch_dict[f])
    for f in list(sm_dict.values())[-1]
]

ax_leg.legend(
    handles= legend_elements,
    loc="upper center",
    ncol=ncol,
    frameon=True
)

data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_deviation_design_r2_smae_SI.pdf"), dpi=300)

In [ ]:
# MAIN: Plot r2 of normalized design deviation for all capacities and feas. preds.
plt.close('all')

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

hatch_dict = {
   'noFP': r'\\',
   'LL' : '--',
   'OC' : 'oo',
   'OC_0.05' : '//',
   'OC_0.1': "..",
   'CH': ''
}
# Set up the plot
fig = plt.figure(figsize=(get_latex_textwidth(), 4.4), constrained_layout=True)
gs = fig.add_gridspec(
    nrows=3, ncols=1,
    height_ratios=[0.10, 0.45, 0.45],
    # width_ratios=[2.5, 3, 2.5]
)

ax_leg = fig.add_subplot(gs[0])
axs = [fig.add_subplot(gs[1]), fig.add_subplot(gs[2])]
color_list = [set_rwth_color(colors[i], 75) for i in sm_dict_N]
WIDTH = 0.87

n_members_per_group = len(sm_dict_N.items())
group_positions = np.array([-5/12, -3/12, -1/12, 1/12, 3/12, 5/12]) * WIDTH

for j, process in enumerate(processes):

    # collect ylims:
    ylim_max_list = []
    data_rmse = []
    data_r2 = []
    data_smae = []

    for i, (feas_pred, data_df) in enumerate(sm_dict_N.items()):
        data_r2.append(r2_score(y_true = df_benchmark[process], y_pred = data_df[process]))
        data_rmse.append(
            np.sqrt(
                calc_mse(
                    y_true = df_benchmark[process] / pot_cap[process],
                    y_pred = data_df[process] / pot_cap[process]).mean()
            )
        )
        data_smae.append(
            (
                calc_mae(
                    y_true = df_benchmark[process],
                    y_pred = data_df[process]
                    ) / pot_cap[process]# * 100
#                    ) / (df_benchmark[process].max() - df_benchmark[process].min())# * 100

                    ).mean()
        )


    axs[0].bar(j + group_positions, data_r2, color=color_list, width=WIDTH/6,
           zorder = 3, edgecolor = 'k', hatch=[hatch_dict[f] for f in sm_dict_N])

    axs[1].bar(j + group_positions, np.array(data_smae)*100, color=color_list, width=WIDTH/6,
           zorder = 2.3, edgecolor = 'k', hatch=[hatch_dict[f] for f in sm_dict_N])

# R2
ax = axs[0]
ax.yaxis.set_major_locator(MultipleLocator(0.5))
ax.yaxis.grid(True, which='major')
ax.yaxis.set_minor_locator(MultipleLocator(0.1))
ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
ax.set_title(rf"coefficient of determination $R^2$")
ax.set_ylabel("$R^2$")
ax.set_ylim(-0.05,1)

# RMSE
ax = axs[1]
ax.yaxis.set_major_locator(MultipleLocator(10))
ax.yaxis.grid(True, which='major')
ax.yaxis.set_minor_locator(MultipleLocator(2))
ax.yaxis.grid(True, which='minor', linestyle='-', linewidth=0.3)
# ax.set_title(rf"root mean square error RMSE")
# ax.set_ylabel(r"RMSE in \%")
ax.set_title(rf"standardized mean absolute error SMAE")
ax.set_ylabel(r"SMAE in \%")
ax.set_ylim(-45/20,45)

for ax in axs:
    ax.set_xticks(range(len(processes)))
    ax.set_xticklabels([ process_name_dict[process] for process in processes])

# Legend:
ax_leg.axis("off")
ncol = n_members_per_group
legend_elements = [
    mpatches.Patch(
        facecolor=set_rwth_color(colors[f], 75),
        edgecolor='k', label=labels[f], hatch = hatch_dict[f])
    for f in sm_dict_N
]

ax_leg.legend(
    handles= legend_elements,
    loc="upper center",
    ncol=ncol,
    frameon=True
)



data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_deviation_design_r2_smae.pdf"), dpi=300)

In [ ]:
# Calc share of OPEX and CAPEX in TAC:

share_OPEX =df_benchmark['OPEX'] / df_benchmark['TAC']
share_CAPEX =  df_benchmark['CAPEX'] / df_benchmark['TAC']

print(f"mean OPEX share = {share_OPEX.mean()* 100:3.2f} %")
print(f"mean CAPEX share = {share_CAPEX.mean()* 100:3.2f} %")

fig, ax = plt.subplots(1,1)

ax.hist(share_OPEX * 100, bins=100)

ax.set_xlabel("share OPEX in %")

### **Step II.5**: Check SADO predictions against fully resolved time series

In [ ]:
# Load Data:

import numpy as np
import matplotlib.pyplot as plta
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.ticker import MultipleLocator
from core.helpers import set_rwth_color

IDX_MIN = 800

value_of_lostload = list(get_value_of_lost_load(DATA_IN, PROJECT_FILE).values())[0]

stats_dicts = {
    '5'   : {'days': 5,   'project': 'project_ID14_5_test365.yaml',   'idx_min': IDX_MIN},
    '10'  : {'days': 10,  'project': 'project_ID14_10_test365.yaml',   'idx_min': IDX_MIN},
    '50'  : {'days': 50,  'project': 'project_ID14_50_test365.yaml',   'idx_min': IDX_MIN},
    '100' : {'days': 100, 'project': 'project_ID14_100_test365.yaml', 'idx_min': IDX_MIN},
}
hsm_pred_dict = {}
benchmark_dict = {}
sm_dict = {}

for key, stat_dict in stats_dicts.items():
    idx_min = stat_dict['idx_min']
    project = stat_dict['project']
    data_out = get_data_out(DATA_IN, project)

    hsm_pred_dict[f'LL_{key}'] =  pd.read_csv(
        os.path.join(data_out,  'design_hybrid_sm_lostload_filtered_max_oc.csv'))[idx_min:]
    
    hsm_pred_dict[f'OC_{key}'] =  pd.read_csv(
        os.path.join(data_out,  'design_hybrid_sm_overcapacity_filtered_max_oc.csv'))[idx_min:]

    hsm_pred_dict[f'OC_{key}_5e-2'] =  pd.read_csv(
        os.path.join(data_out,  'design_hybrid_sm_overcapacity_5e-2_filtered_max_oc.csv'))[idx_min:]

    hsm_pred_dict[f'OC_{key}_1e-1'] =  pd.read_csv(
        os.path.join(data_out,  'design_hybrid_sm_overcapacity_1e-1_filtered_max_oc.csv'))[idx_min:]

    hsm_pred_dict[f'CH_{key}'] =  pd.read_csv(
        os.path.join(data_out,  'design_hybrid_sm_convexhull_filtered_max_oc.csv'))[idx_min:]

    hsm_pred_dict[f'noFP_{key}'] =  pd.read_csv(
        os.path.join(data_out,  'design_hybrid_sm_noFP_filtered_max_oc.csv'))[idx_min:]

# add TAC_IST to all dicts:
for key, df in hsm_pred_dict.items():
    # Assert same length:
    if 10000 != len(df):
        raise ValueError(
            f"Length of sado prediction do not match 10000 for {key}"
            f" (10000 != {len(df)})"
        )


for key, stat_dict in stats_dicts.items():
    sm_dict[key] = {
        'noFP'    : hsm_pred_dict[f'noFP_{key}'],
        'LL'      : hsm_pred_dict[f'LL_{key}'],
        'OC'      : hsm_pred_dict[f'OC_{key}'],
        'OC_0.05' : hsm_pred_dict[f'OC_{key}_5e-2'],
        'OC_0.1'  : hsm_pred_dict[f'OC_{key}_1e-1'],
        'CH'      : hsm_pred_dict[f'CH_{key}'] ,
    }

colors = {
    'LL': 'orange',
    'OC': 'blue',
    'OC_0.05': 'green',
    'OC_0.1': 'petrol',
    'CH': 'magenta',
    'BM': 'black',
    'noFP': 'red'
    }

labels = {
    'LL': r'ENS',
    'OC': r'OC, 0\%',
    'OC_0.05': r'OC, 5\%',
    'OC_0.1': r'OC, 10\%',
    'CH': 'CH',
    'BM': 'BM',
    'noFP': 'none'
}

In [ ]:
import matplotlib.transforms as mtransforms
from core.plotting_helpers import row_major_for_legend

def get_fraction_of_infeasible_samples(df, tol=1e-13):
    return len(df[df['overcapacity'] < -tol]) / len(df) * 100

def make_figure(panel_kinds, outpath):
    """
    panel_kinds: list like ['oc_full','oc_zoom','infeasible'] (SI)
                       or ['oc_zoom','infeasible'] (main)
    """
    plt.close('all')

    plt.rcParams['text.usetex'] = True
    plt.rcParams['font.family'] = 'serif'

    # --- Figure layout : choose grid based on panel count ---
    if len(panel_kinds) == 3:
        fig = plt.figure(figsize=(6.8425, 6), constrained_layout=True)
        gs = fig.add_gridspec(
            nrows=2, ncols=3,
            height_ratios=[0.06, 0.94],
            width_ratios=[2.5, 3, 2.5]
        )
        ax_leg = fig.add_subplot(gs[0, :])
        axs = [fig.add_subplot(gs[1, 0]),
               fig.add_subplot(gs[1, 1]),
               fig.add_subplot(gs[1, 2])]

    elif len(panel_kinds) == 2:
        fig = plt.figure(figsize=(6.8425, 6), constrained_layout=True)
        gs = fig.add_gridspec(
            nrows=2, ncols=2,
            height_ratios=[0.06, 0.94],
            width_ratios=[3, 2.5]   # zoom + infeasible
        )
        ax_leg = fig.add_subplot(gs[0, :])
        axs = [fig.add_subplot(gs[1, 0]),
               fig.add_subplot(gs[1, 1])]
    else:
        raise ValueError("panel_kinds must have length 2 or 3")

    n_members_per_group = len(list(sm_dict.values())[1])
    n_groups = len(sm_dict)

    group_positions = get_grouped_positions(n_groups, n_members_per_group, 0.7)
    center_positions = get_center_positions(n_groups, n_members_per_group, 0.7)

    # --- Main plotting loop: switch by panel_kinds[i] ---
    for i, ax in enumerate(axs):
        kind = panel_kinds[i]

        n_days = []
        for n_group, (key, subdict) in enumerate(reversed(sm_dict.items())):
            n_days.append(key)
            candidates = [name for name in subdict.keys() if name != 'days']

            for n_member, candidate in enumerate(reversed(candidates)):
                df = subdict[candidate]
                pos = group_positions[n_group*n_members_per_group + n_member]

                if kind in {'oc_full', 'oc_zoom'}:
                    data = df['overcapacity'].values * 100
                    # print(f"{key} | {candidate}: {np.median(data):.2f} %")

                    p95 = plot_custom_violin(ax, data, pos, color=colors[candidate], s=36)

                    if len(panel_kinds) == 2 and data.mean() > 30:
                        X0 = 9
                        X1 = 19
                        ax.annotate(
                            "",
                            xy=(X1, pos), xytext=(X0,pos),
                            arrowprops=dict(arrowstyle="->", linewidth=1.1)
                        )
                                
                        # Place centered text above the arrow
                        ax.text(
                            (X0 + X1)/2,        # center in x
                            pos + 0.15,       # small offset above arrow
                            rf"mean at {data.mean():3.0f} \%",
                            ha="center",
                            va="bottom"
                        )
                elif kind == 'infeasible':
                    frac_infeasible = get_fraction_of_infeasible_samples(df)
                    ax.barh(
                        pos, frac_infeasible, height=0.7,
                        color=set_rwth_color(colors[candidate], 50),
                        edgecolor=set_rwth_color(colors[candidate], 100)
                    )
                    ax.text(
                        45, pos-0.135,
                        rf'{frac_infeasible:.2f}\%',
                        va='center', fontsize=8, ha='right',
                        color='black'
                    )

        # Plot group separations
        trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
        ax.hlines(
            [(center_positions[k] + center_positions[k+1]) / 2
             for k in range(len(center_positions) - 1)],
            xmin=0, xmax=1, color='grey', alpha=0.6, transform=trans
        )

    # --- Legend elements ---
    legend_elements = [
        mpatches.Patch(
            facecolor=set_rwth_color(colors[candidate], 50),
            edgecolor=set_rwth_color(colors[candidate], 100),
            label=labels[candidate]
        )
        for candidate in candidates
    ] + [
        Line2D([0], [0], color='k', lw=4, linestyle='-',
               label=r'25$^\mathrm{th}$--75$^\mathrm{th}$'),
        Line2D([0], [0], color='k', lw=2, linestyle='-',
               label=r'5$^\mathrm{th}$--95$^\mathrm{th}$'),
        Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='k',
               markerfacecolor='white', markersize=np.sqrt(50), label='mean'),
        Line2D([0], [0], color='k', lw=0, marker='^', markeredgecolor='k',
               markerfacecolor='white', markersize=np.sqrt(50), label='median'),
        Line2D([0], [0], color='k', lw=0, marker='o', markeredgecolor='grey',
               markerfacecolor='grey', markersize=np.sqrt(1), label='outlier'),
    ]

    # --- Right axes (twinx), y ticks/labels ---
    axs_r = []
    for ax in axs:
        ax.set_ylim(group_positions[0]-1, group_positions[-1]+1)
        ax.set_yticks(group_positions)

        ax_r = ax.twinx()
        ax_r.set_ylim(ax.get_ylim())
        ax_r.set_yticks(center_positions)
        axs_r.append(ax_r)

        ax.set_yticklabels([])
        ax_r.set_yticklabels([])

    # Predictor labels go on first panel
    axs[0].set_yticklabels([labels[c] for c in candidates][::-1] * 4)
    axs[0].set_ylabel("feasibility predictor")

    # Typical-days labels go on last panel
    axs_r[-1].set_ylabel("number of typical days")
    axs_r[-1].set_yticklabels(
        # [f'{days} days\napplied to\n365 days'
        # ['surrogate models\ntrained on\n\\textbf{' + str(days) + 'days}\n tested on\n365 days'
        ['\\textbf{' + str(days) + ' days}\n tested on\n365 days'
         for days in n_days]
        )

    # --- X labels / limits by panel kind  ---
    for ax, kind in zip(axs, panel_kinds):
        if kind == 'oc_full':
            ax.set_xlabel(r"overcapacity $\varepsilon^\mathrm{OC}$ in \%")
            ax.xaxis.set_major_locator(MultipleLocator(200))
            ax.xaxis.grid(True, which='major')
            ax.xaxis.set_minor_locator(MultipleLocator(50))
        elif kind == 'oc_zoom':
            # ax.set_xlabel("overcapacity $\\varepsilon^\\mathrm{OC}$ in \\%\ntested in fully resolved background model")
            ax.set_xlabel(r"overcapacity $\varepsilon^\mathrm{OC}$ in \%")
            ax.set_xlim(-10, 20)
            ax.xaxis.grid(True, which='major')
            ax.xaxis.set_minor_locator(MultipleLocator(1))
        elif kind == 'infeasible':
            ax.set_xlabel(r"infeasible samples in \%")
            ax.xaxis.grid(True, which='major')
            ax.xaxis.set_minor_locator(MultipleLocator(5))

        ax.set_axisbelow(True)

    # --- Legend axes ---
    ax_leg.axis('off')
    ncol = len(candidates)
    ax_leg.legend(
        handles=row_major_for_legend(legend_elements, ncol),
        loc='upper center',
        frameon=True,
        ncol=ncol
    )

    plt.savefig(outpath, dpi=300)

# -------------------------
# Build BOTH figures at once
# -------------------------
data_out = get_data_out(DATA_IN, "project_ID14_100_test365.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

# Main text: drop full-range panel
make_figure(
    panel_kinds=['oc_zoom', 'infeasible'],
    outpath=os.path.join(data_out, "plots", "fig05_violin_oc_SI_test365.pdf")
)

# # SI: keep full-range panel
# make_figure(
#     panel_kinds=['oc_full', 'oc_zoom', 'infeasible'],
#     outpath=os.path.join(data_out, "plots", "fig05_violin_oc_SI_test365.pdf")
# )


# **(III)**: Compare compuational effort

## **Step III.1**: Compare computational effort of operational optimizations

In [ ]:
data_in = DATA_IN
project = 'project_ID14_100.yaml'
op_test_csv_name = 'op_var_100000_test.csv'
maxoc_csv_name = 'op_var_max_overcapacity_100000.csv'


In [ ]:
n_days :list  = [5, 10, 50, 100]

CSV_NAME_OPERAT = 'op_var_100000.csv'
CSV_NAME_MAXOC  = 'op_var_max_overcapacity_100000.csv'

df_sm_times = pd.read_csv(
    os.path.join(get_data_out(DATA_IN, "project_ID14_100.yaml"), "ann_prediction_times.csv"))

dict_all = {
    model : {
        'time'      : {},
        'min'       : [],
        'max'       : [],
        'mean'      : [],
        'conf_5'    : [],
        'conf_95'   : [],
        'conf_25'   : [],
        'conf_75'   : [],
        'conf_1'    : [],
        'conf_99'   : [],
    } for model in [
        'bg_max_oc', # Background model max overcapacity
        'bg_operat', # Background model operational optimization
        'sm_opex', # Surrogate model for OPEX
        'sm_oc',  # Surrogate model for OC
        'sm_ens', # Surrogate model for ENS
    ]
}

# Read times:
for n_day in n_days:
    data_out = get_data_out(DATA_IN, f"project_ID14_{n_day}.yaml")

    # Read times for overcapacity background model
    df = pd.read_csv(os.path.join(data_out, CSV_NAME_MAXOC))
    dict_all['bg_max_oc']['time'][n_day] = df['solver_time'].values

    # Read times for opex background model
    df = pd.read_csv(os.path.join(data_out, CSV_NAME_OPERAT))
    dict_all['bg_operat']['time'][n_day] = df['solver_time'].values

    # Read times for sm_opex
    dict_all['sm_opex']['time'][n_day] = df_sm_times[f"N{n_day}_OPEX_woLL"].values

    # Read times for sm_oc
    dict_all['sm_oc']['time'][n_day] = df_sm_times[f"N{n_day}_OC"].values

    # Read times for sm_ens
    dict_all['sm_ens']['time'][n_day] = df_sm_times[f"N{n_day}_ENS"].values

# Add statistics:
for n_day in n_days:
    for model_dict in dict_all.values():

        time = model_dict['time'][n_day]
    
        model_dict['min'].append( np.min(time))
        model_dict['max'].append(     np.max(time))
        model_dict['mean'].append(    np.mean(time))     
        model_dict['conf_5'].append(  np.percentile(time, 5))
        model_dict['conf_95'].append( np.percentile(time, 95))
        model_dict['conf_25'].append( np.percentile(time, 25))
        model_dict['conf_75'].append( np.percentile(time, 75))
        model_dict['conf_1'].append( np.percentile(time, 1))
        model_dict['conf_99'].append( np.percentile(time, 99))

In [ ]:
# MAIN TEXT

plt.close('all')
MARKERSIZE_S = 25
ALPHA_FACE = 0.12
ALPHA_EDGE = 0.8
LINEWIDTH_EDGE = 0.75

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

fig, ax = plt.subplots(1,1, figsize=(3.29672671,2.2), constrained_layout=True)

# plot mean of OPEX Background mdoel
ax.plot(range(len(n_days)), dict_all['bg_operat']['mean'], marker='o', zorder=3,
        label='original operational model', color=set_rwth_color('black', 100), markersize = np.sqrt(MARKERSIZE_S))

# plot mean of MAXOC Background model
ax.plot(range(len(n_days)), dict_all['bg_max_oc']['mean'], marker='s', zorder=3,
        label='original overcapacity model', color=set_rwth_color('green', 100),
        markersize = np.sqrt(MARKERSIZE_S))

# plot mean of sm OPEX
ax.plot(range(len(n_days)), dict_all['sm_opex']['mean'], marker='v',  zorder=3,
    label='cost, ENS, and OC predictor', color=set_rwth_color('petrol', 100),
    markersize = np.sqrt(MARKERSIZE_S))

ax.set_xticks(range(len(n_days)))
ax.set_xticklabels([f'{days} days' for days in n_days])
ax.set_xlabel('number of typical days')
ax.set_ylabel('mean evaluation time in s')
ax.set_yscale('log')
ax.grid(which='both')

ax.legend(
    loc="center",
    bbox_to_anchor=(1.5, 0.005),
    bbox_transform=ax.transData,
    framealpha=1
)

# Save plot:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig08.pdf"), dpi=300, bbox_inches='tight') #fig05_osm_prediction_time


In [ ]:
# SUPPORTING INFORMATION

plt.close('all')
MARKERSIZE_S = 36
ALPHA_FACE = 0.12
ALPHA_EDGE = 0.8
LINEWIDTH_EDGE = 0.75
LINESTYLE_25_75 = '-.'
LINESTYLE_MINMAX = '--'

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

fig, ax = plt.subplots(1,1, figsize=(get_latex_textwidth(), 4), constrained_layout=True)

# plot mean of OPEX Background mdoel
ax.plot(range(len(n_days)), dict_all['bg_operat']['mean'], marker='o', zorder=3,
        label='original operational model (mean time)', color='k', markersize = np.sqrt(MARKERSIZE_S))
ax.plot(range(len(n_days)), dict_all['bg_operat']['min'], marker='o', zorder=3,
        color='k', markersize = np.sqrt(MARKERSIZE_S), linestyle=LINESTYLE_MINMAX)
ax.plot(range(len(n_days)), dict_all['bg_operat']['max'], marker='o', zorder=3,
        color='k', markersize = np.sqrt(MARKERSIZE_S), linestyle=LINESTYLE_MINMAX)

# plot mean of MAXOC Background model
ax.plot(range(len(n_days)), dict_all['bg_max_oc']['mean'], marker='s', zorder=3,
        label='original overcapacity model (mean time)', color=set_rwth_color('green'),
        markersize = np.sqrt(MARKERSIZE_S))
ax.plot(range(len(n_days)), dict_all['bg_max_oc']['min'], marker='s', zorder=3,
        color=set_rwth_color('green'), linestyle=LINESTYLE_MINMAX,
        markersize = np.sqrt(MARKERSIZE_S))
ax.plot(range(len(n_days)), dict_all['bg_max_oc']['max'], marker='s', zorder=3,
        color=set_rwth_color('green'), linestyle=LINESTYLE_MINMAX,
        markersize = np.sqrt(MARKERSIZE_S))

# plot mean of sm OPEX
ax.plot(range(len(n_days)), dict_all['sm_opex']['mean'], marker='v',  zorder=3,
    label='cost, ENS, and OC predictor (mean time)', color=set_rwth_color('petrol'),
    markersize = np.sqrt(MARKERSIZE_S))
ax.plot(range(len(n_days)), dict_all['sm_opex']['min'], marker='v',  zorder=3,
    color=set_rwth_color('petrol'), linestyle=LINESTYLE_MINMAX,
    markersize = np.sqrt(MARKERSIZE_S))
ax.plot(range(len(n_days)), dict_all['sm_opex']['max'], marker='v',  zorder=3,
    color=set_rwth_color('petrol'), linestyle=LINESTYLE_MINMAX,
    markersize = np.sqrt(MARKERSIZE_S))

ax.set_xticks(range(len(n_days)))
ax.set_xticklabels([f'{days} days' for days in n_days])
ax.set_xlabel('number of typical days')
ax.set_ylabel('evaluation time in s')
ax.set_ylim([2e-4, 1e2])
ax.set_yscale('log')
ax.grid(which='both')

# dummy line for legend:

ax.plot([1, 2], [1e4, 1e4], zorder=3,
        color= set_rwth_color('black', 75), linestyle=LINESTYLE_MINMAX,
        label='min and max values')

ax.legend(
    loc="best",
    framealpha = 1
    # bbox_to_anchor=(2.2, 0.005),
    # bbox_transform=ax.transData
)

# Save plot:
data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)

plt.savefig(
    os.path.join(data_out, "plots", "fig05_osm_prediction_time_SI.pdf"), dpi=300, bbox_inches='tight')


## **Step III.2**: Compare computational effort of design optimizations

In [ ]:
n_days = [5, 10, 50, 100]

sm_design_dict = {
    n: {
        'oc' : pd.read_csv(
            os.path.join(get_data_out(DATA_IN, f'project_ID14_{n}.yaml'), 'design_hybrid_sm_overcapacity.csv')),
        'oc_0.1' : pd.read_csv(
            os.path.join(get_data_out(DATA_IN, f'project_ID14_{n}.yaml'), 'design_hybrid_sm_overcapacity_1e-1.csv')),
        'oc_0.05' : pd.read_csv(
            os.path.join(get_data_out(DATA_IN, f'project_ID14_{n}.yaml'), 'design_hybrid_sm_overcapacity_5e-2.csv')),
        'ens' : pd.read_csv(
            os.path.join(get_data_out(DATA_IN, f'project_ID14_{n}.yaml'), 'design_hybrid_sm_lostload.csv')),
        'ch' : pd.read_csv(
            os.path.join(get_data_out(DATA_IN, f'project_ID14_{n}.yaml'), 'design_hybrid_sm_convexhull.csv')),
        'none' : pd.read_csv(
            os.path.join(get_data_out(DATA_IN, f'project_ID14_{n}.yaml'), 'design_hybrid_sm_noFP.csv')),
        'days': n,
        'opt':pd.read_csv(
            os.path.join(get_data_out(DATA_IN, f'project_ID14_{n}.yaml'), 'design_10800.csv')),
            }
        for n in n_days
        }

In [ ]:
sado_types = [sado_type for sado_type in sm_design_dict[n_days[0]] if sado_type != 'days']

time_sado = {sado_type : {
    'min': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'max': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'mean': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'conf_5': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'conf_95': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'conf_25': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'conf_75': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'conf_1': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'conf_99': np.zeros_like(range(len(sm_design_dict)), dtype=float),
    'days': np.zeros_like(range(len(sm_design_dict)), dtype=float),
} for sado_type in sado_types}

n_days_design = np.array([sm_design_dict['days'] for sm_design_dict in sm_design_dict.values()])

for i, sub_dict in enumerate(sm_design_dict.values()):


    for sado_type in sado_types:

        time_key = 'execution_time' if sado_type != 'opt' else 'solver_time'
        dt_array = sub_dict[sado_type][time_key].values

        time_sado[sado_type]['min'][i]     = np.min(dt_array)
        time_sado[sado_type]['max'][i]     = np.max(dt_array)
        time_sado[sado_type]['mean'][i]    = np.mean(dt_array)
        time_sado[sado_type]['conf_5'][i]  = np.percentile(dt_array, 5)
        time_sado[sado_type]['conf_95'][i] = np.percentile(dt_array, 95)
        time_sado[sado_type]['conf_25'][i] = np.percentile(dt_array, 25)
        time_sado[sado_type]['conf_75'][i] = np.percentile(dt_array, 75)
        time_sado[sado_type]['conf_1'][i] = np.percentile(dt_array, 1)
        time_sado[sado_type]['conf_99'][i] = np.percentile(dt_array, 99)

In [ ]:
plt.close('all')
MARKERSIZE_S = 25
MARKERSIZE_S_small = MARKERSIZE_S / 2
ALPHA_FACE = 0.12
ALPHA_EDGE = 0
LINEWIDTH_EDGE = 0.75
LINESTYLE_EDGE = ':'

plt.rcParams['text.usetex'] = True  
plt.rcParams['font.family'] = 'serif'

def plot_fig(SI: bool=False):
    if SI:
        fig = plt.figure(figsize=(get_latex_textwidth(),5), constrained_layout=True)
        gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.03, 0.93])
    else:
        fig = plt.figure(figsize=(get_latex_columnwidth() ,2.8), constrained_layout=True)
        gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.2, 0.8])

    ax_leg  = fig.add_subplot(gs[0])
    ax      = fig.add_subplot(gs[1])

    # plot mean of optimization
    ax.plot(
        range(len(sm_design_dict)), time_sado['opt']['mean'], marker='o', label='original design model',
        color='k', zorder=3, markersize = np.sqrt(MARKERSIZE_S))
    # plot 5 and 95 percentile of optimization
    if SI:
        ax.fill_between(
            range(len(sm_design_dict)), time_sado['opt']['min'], time_sado['opt']['max'],
            facecolor = to_rgba('k', ALPHA_FACE),
            edgecolor = to_rgba('k', ALPHA_EDGE),
            linewidth = LINEWIDTH_EDGE,
            zorder=2)
        
        ax.plot(
            range(len(sm_design_dict)), time_sado['opt']['min'], marker='o',
            color='k', zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)
        ax.plot(
            range(len(sm_design_dict)), time_sado['opt']['max'], marker='o',
            color='k', zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)

    # plot mean of SADO ENS
    ax.plot(range(len(sm_design_dict)), time_sado['ens']['mean'], marker='s',
            label='SADO with ENS predictor', color=set_rwth_color('orange'), zorder=3,
            markersize = np.sqrt(MARKERSIZE_S))
    # plot 5 and 95 percentile of surrogate model
    if SI:
        ax.fill_between(
            range(len(sm_design_dict)), time_sado['ens']['min'], time_sado['ens']['max'],
            facecolor = to_rgba(set_rwth_color('orange'), ALPHA_FACE),
            edgecolor = to_rgba(set_rwth_color('orange'), ALPHA_EDGE),
            linewidth = LINEWIDTH_EDGE,
            zorder=2)
        
        ax.plot(
            range(len(sm_design_dict)), time_sado['ens']['min'], marker='s',
            color=set_rwth_color('orange'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)
        ax.plot(
            range(len(sm_design_dict)), time_sado['ens']['max'], marker='s',
            color=set_rwth_color('orange'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)

    # plot mean of SADO OC (0%)
    ax.plot(range(len(sm_design_dict)), time_sado['oc']['mean'], marker='v',
            label='SADO with OC predictor', color=set_rwth_color('blue'), zorder=3,
            markersize = np.sqrt(MARKERSIZE_S))
    # plot 5 and 95 percentile of surrogate model
    if SI:
        ax.fill_between(
            range(len(sm_design_dict)), time_sado['oc']['min'], time_sado['oc']['max'],
            facecolor = to_rgba(set_rwth_color('blue'), ALPHA_FACE),
            edgecolor = to_rgba(set_rwth_color('blue'), ALPHA_EDGE),
            linewidth = LINEWIDTH_EDGE,
            zorder=2.2)
        
        ax.plot(
            range(len(sm_design_dict)), time_sado['oc']['min'], marker='v',
            color=set_rwth_color('blue'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)
        ax.plot(
            range(len(sm_design_dict)), time_sado['oc']['max'], marker='v',
            color=set_rwth_color('blue'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)

    # plot mean of SADO CH
    ax.plot(range(len(sm_design_dict)), time_sado['ch']['mean'], marker='^',
            label='SADO with CH predictor', color=set_rwth_color('magenta'), zorder=3,
            markersize = np.sqrt(MARKERSIZE_S))
    # plot 5 and 95 percentile of surrogate model
    if SI:
        ax.fill_between(
            range(len(sm_design_dict)), time_sado['ch']['min'], time_sado['ch']['max'],
            facecolor = to_rgba(set_rwth_color('magenta'), ALPHA_FACE),
            edgecolor = to_rgba(set_rwth_color('magenta'), ALPHA_EDGE),
            linewidth = LINEWIDTH_EDGE,
            zorder=2)
        
        ax.plot(
            range(len(sm_design_dict)), time_sado['ch']['min'], marker='^',
            color=set_rwth_color('magenta'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)
        ax.plot(
            range(len(sm_design_dict)), time_sado['ch']['max'], marker='^',
            color=set_rwth_color('magenta'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)

    # plot mean of no FP
    ax.plot(range(len(sm_design_dict)), time_sado['none']['mean'], marker='x',
            label=r'SADO w/o feas. predictor', color=set_rwth_color('red'), zorder=3,
            markersize = np.sqrt(MARKERSIZE_S))
    # plot 5 and 95 percentile of surrogate model
    if SI:
        ax.fill_between(
            range(len(sm_design_dict)), time_sado['none']['min'], time_sado['none']['max'],
            facecolor = to_rgba(set_rwth_color('red'), ALPHA_FACE),
            edgecolor = to_rgba(set_rwth_color('red'), ALPHA_EDGE),
            linewidth = LINEWIDTH_EDGE,
            zorder=2.2)

        ax.plot(
            range(len(sm_design_dict)), time_sado['none']['min'], marker='x',
            color=set_rwth_color('red'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)
        ax.plot(
            range(len(sm_design_dict)), time_sado['none']['max'], marker='x',
            color=set_rwth_color('red'), zorder=3, markersize = np.sqrt(MARKERSIZE_S_small), linestyle=LINESTYLE_EDGE)

    if SI:
        ax.plot([1, 1], [1e3, 1e3], color = set_rwth_color('black', 75), linestyle=LINESTYLE_EDGE, label='min and max')

    ax.set_xticks(range(len(sm_design_dict)))
    ax.set_xticklabels([f'{days} days' for days in n_days_design])
    ax.set_xlabel('number of typical days')
    if SI:
        ax.set_ylabel('evaluation time in s')
    else:
        ax.set_ylabel('mean evaluation time in s')
    ax.set_yscale('log')
    if SI:
        ax.set_ylim((1e-2, 2e2))
    else:
        ax.set_ylim((1e-1, 1e2))
    # ax.set_ylim((0, 2))
    # ax.legend(loc='best', ncol = 2)
    ax.grid(which='both')


    # Legend
    ax_leg.axis("off")
    handles, labels = ax.get_legend_handles_labels()
    ax_leg.legend(
        handles, labels,
        loc="upper center",
        ncol=2 if SI else 1,
        frameon=True,
        fontsize=8
    )

    # Save plot:
    data_out = get_data_out(DATA_IN, "project_ID14_100.yaml")
    os.makedirs(os.path.join(data_out, "plots"), exist_ok=True)
    
    if SI:
        plt.savefig(
            os.path.join(data_out, "plots", "fig05_sado_prediction_time_SI.pdf"), dpi=300, bbox_inches='tight')
    else:
        plt.savefig(
            os.path.join(data_out, "plots", "fig14.pdf"), dpi=300, bbox_inches='tight')
        
plot_fig(SI=True)
plot_fig(SI=False)

In [ ]:

print("CH is X times slower")
mean_time_CH = time_sado['ch']['mean']
for feas_pred in ['oc', 'ens', 'none']:
    mean_time = time_sado[feas_pred]['mean']
    relative_mean = mean_time_CH / mean_time

    print(f"{feas_pred}: {relative_mean}")

print("\nSADO is X times faster than background model:")

mean_time_opt = time_sado['opt']['mean']
for feas_pred in ['ch', 'oc', 'ens', 'none']:
    mean_time = time_sado[feas_pred]['mean']
    relative_mean = mean_time_opt / mean_time

    print(f"{feas_pred}: {relative_mean}")



In [ ]:
time_sado.keys()

In [ ]:
# Check ratio of mean prediction times.
for n_day in n_days:
    